### Configurations

In [ ]:
%%capture
# no Colab instala LaTeX e cmcrameri; no PC instale antes (ver README)
import os, subprocess, sys
if os.path.exists("/content"):
    subprocess.run("apt-get update -qq && apt-get install -y -qq texlive texlive-latex-extra texlive-fonts-recommended dvipng cm-super", shell=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cmcrameri"])


In [ ]:
import os
import glob
import time
import uuid
import copy
import random
import itertools
import colorsys

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as colors
import matplotlib.patheffects as pe
import matplotlib.ticker as ticker

from itertools import permutations

from scipy import integrate
from scipy.integrate import quad
from scipy.signal import find_peaks, peak_widths
from scipy.special import (
    eval_legendre,
    lpmv,
    spherical_jn,
    spherical_yn,
)

from matplotlib.colors import (
    LogNorm,
    LinearSegmentedColormap,
    BoundaryNorm,
    ListedColormap,
)

from matplotlib.ticker import (
    ScalarFormatter,
    LogLocator,
    LogFormatter,
    NullFormatter,
)

from matplotlib.patches import Rectangle
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.axes_grid1 import make_axes_locatable

try:
    import cmcrameri.cm as cmc
except Exception:
    cmc = None


plt.rcParams.update({
    "text.usetex"   : True,
    "figure.dpi"    : 600,
    "font.family" : "sans-serif",
})

In [ ]:
# pasta com a saída do gerador (a mesma passada para os scripts de relações)
import os
DADOS = os.environ.get("CORESHELL_DADOS", os.path.abspath(os.path.join("..", "dados")))
if os.path.exists("/content"):   # Colab: lê do Drive
    from google.colab import drive
    drive.mount("/content/drive")
    DADOS = "/content/drive/MyDrive/coreshell_final/resultados_mix"
print("Dados em:", DADOS)


### Integrals

\begin{equation}
\begin{split}
F_{jj}(x_B,y_B)
&=
\left[
x^3
\left(
j_\ell^2(x)
-
j_{\ell-1}(x)j_{\ell+1}(x)
\right)
\right]_{x_B}^{y_B},
\\
F_{yy}(x_B,y_B)
&=
\left[
x^3
\left(
y_\ell^2(x)
-
y_{\ell-1}(x)y_{\ell+1}(x)
\right)
\right]_{x_B}^{y_B},
\\
F_{jy}(x_B,y_B)
&=
\left[
\frac{x^3}{2}
\left(
2j_\ell(x)y_\ell(x)
-
j_{\ell+1}(x)y_{\ell-1}(x)
-
j_{\ell-1}(x)y_{\ell+1}(x)
\right)
\right]_{x_B}^{y_B},
\end{split}
\end{equation}

\begin{equation}
\begin{split}
B_{jj}(x_B,y_B)
&=
\left[
x^2
\left(
j_\ell(x)\,j'_\ell(x)
\right)
\right]_{x_B}^{y_B},
\\
B_{yy}(x_B,y_B)
&=
\left[
x^2
\left(
y_\ell(x)\,y'_\ell(x)
\right)
\right]_{x_B}^{y_B},
\\
B_{jy}(x_B,y_B)
&=
\left[
x^2
\left(
j_\ell(x)\,y'_\ell(x)
\right)
\right]_{x_B}^{y_B},
\end{split}
\end{equation}

In [ ]:
def Fjj(l, xB, yB):
    def F(x):
        return x**3 * (J(l, x)**2 - J(l - 1, x) * J(l + 1, x))
    return F(yB) - F(xB)


def Fyy(l, xB, yB):
    def F(x):
        return x**3 * (Y(l, x)**2 - Y(l - 1, x) * Y(l + 1, x))
    return F(yB) - F(xB)


def Fjy(l, xB, yB):
    def F(x):
        return 0.5 * x**3 * (2 * J(l, x) * Y(l, x)- J(l + 1, x) * Y(l - 1, x)- J(l - 1, x) * Y(l + 1, x))
    return F(yB) - F(xB)


def Bjj(l, xB, yB):
    def B(x):
        return x**2 * J(l, x) * dJ(l, x)

    return B(yB) - B(xB)


def Byy(l, xB, yB):
    def B(x):
        return x**2 * Y(l, x) * dY(l, x)
    return B(yB) - B(xB)


def Bjy(l, xB, yB):
    def B(x):
        return x**2 * J(l, x) * dY(l, x)
    return B(yB) - B(xB)


def FA(l, xA):
    return 0.5 * xA**3 * (J(l, xA)**2 - J(l - 1, xA) * J(l + 1, xA))


def BA(l, xA):
    return xA**2 * J(l, xA) * dJ(l, xA)


def ell_max(x0_max):
    x0_max = float(x0_max)
    return int(3 + x0_max + 4.05 * x0_max**(1/3))


def get_lmax_core_shell(xM_vals, delta, c0, c1, c2, pad_core=0, pad_shell=0):
    xM_vals = np.asarray(xM_vals, dtype=float)
    xM_max = np.max(xM_vals)

    xA_max = (c0 / c1) * xM_max
    xB_max = (c0 / c2) * xM_max
    yB_max = (1.0 + delta) * xB_max

    lmax_core = ell_max(xA_max) + int(pad_core)
    lmax_shell = ell_max(yB_max) + int(pad_shell)

    return lmax_core, lmax_shell

### Bessel spherical functions




In [ ]:
def J(n, x):
    if n >= 0:
        return spherical_jn(n, x)
    m = -n - 1
    return (-1)**(m+1) * spherical_yn(m, x)

def Y(n, x):
    if n >= 0:
        return spherical_yn(n, x)
    m = -n - 1
    return (-1)**m * spherical_jn(m, x)

def dJ(n, x):
    if n >= 0:
        return spherical_jn(n, x, derivative=True)
    m = -n - 1
    return (-1)**(m+1) * spherical_yn(m, x, derivative=True)

def dY(n, x):
    if n >= 0:
        return spherical_yn(n, x, derivative=True)
    m = -n - 1
    return (-1)**m * spherical_jn(m, x, derivative=True)

def H(n, x):
    if n >= 0:
        return spherical_jn(n, x) + 1j * spherical_yn(n, x)
    m = -n - 1
    return 1j * (-1)**m * (spherical_jn(m, x) + 1j * spherical_yn(m, x))

def dH(n, x):
    if n >= 0:
        return spherical_jn(n, x, derivative=True) + 1j * spherical_yn(n, x, derivative=True)
    m = -n - 1
    return 1j * (-1)**m * (
        spherical_jn(m, x, derivative=True) + 1j * spherical_yn(m, x, derivative=True)
    )

### Coefficients

In [ ]:
def D_l_coreshell(l, xA, xB, yM, yB, rho0, rho1, rho2, c0, c1, c2):
    rho_MB = rho0 / rho2
    rho_AB = rho1 / rho2
    k_MB = c2 / c0
    k_AB = c2 / c1

    A = -J(l, yB)
    B = -Y(l, yB)
    F = rho_MB * H(l, yM)

    G = -dJ(l, yB)
    HH = -dY(l, yB)
    JJ = k_MB * dH(l, yM)

    L = -rho_AB * J(l, xA)
    M = J(l, xB)
    P = Y(l, xB)

    Q = -k_AB * dJ(l, xA)
    R = dJ(l, xB)
    S = dY(l, xB)

    return (
        (L * R - M * Q) * (F * HH - B * JJ)
        + (L * S - P * Q) * (A * JJ - F * G)
    )

def coeffs(l, xA, xB, yM, yB, rho0, rho1, rho2, c0, c1, c2):
    rho_MB = rho0 / rho2
    rho_AB = rho1 / rho2

    k_MB = c2 / c0
    k_AB = c2 / c1

    A = -J(l, yB)
    B = -Y(l, yB)
    F = rho_MB * H(l, yM)

    G = -dJ(l, yB)
    HH = -dY(l, yB)
    JJ = k_MB * dH(l, yM)

    L = -rho_AB * J(l, xA)
    M = J(l, xB)
    P = Y(l, xB)

    Q = -k_AB * dJ(l, xA)
    R = dJ(l, xB)
    S = dY(l, xB)


    D = ((L*R - M*Q) * (F*HH - B*JJ)+ (L*S - P*Q) * (A*JJ - F*G))

    X = 1j * rho_MB * k_MB / (yM**2)

    b_l = (M*S - P*R) * X / D
    c_l = (P*Q - L*S) * X / D
    d_l = (L*R - M*Q) * X / D

    return b_l, c_l, d_l

### Core energy

For the core, the potential and kinetic energies are evaluated directly from the
closed radial expressions. With $x_A=k_Aa$ and $Z_A=\rho_Ac_A$,

\begin{equation}
\frac{2 f_0 W_P^{(A)}}{Z_A |A|^2}
=
\sum_{\ell=0}^{\infty}
|b_\ell|^2(2\ell+1)
\frac{x_A^3}{2}
\left[
j_\ell^2(x_A)
-
j_{\ell-1}(x_A)j_{\ell+1}(x_A)
\right],
\end{equation}

\begin{equation}
\frac{2 f_0 W_K^{(A)}}{Z_A |A|^2}
=
\sum_{\ell=0}^{\infty}
|b_\ell|^2(2\ell+1)
\left\{
\frac{x_A^3}{2}
\left[
j_\ell^2(x_A)
-
j_{\ell-1}(x_A)j_{\ell+1}(x_A)
\right]
+
x_A^2 j_\ell(x_A)j_\ell'(x_A)
\right\}.
\end{equation}

Thus, the total core energy is simply

\begin{equation}
W_T^{(A)} = W_P^{(A)} + W_K^{(A)} .
\end{equation}

This form avoids the numerical evaluation of the auxiliary radial integrals used
in the previous implementation.

In [ ]:
def W_core_parts_vs_xM(xM_vals, delta, rho0, rho1, rho2, c0, c1, c2, lmax_core=None, pad_core=0, lmax_cap=None):

    xM_vals = np.asarray(xM_vals, dtype=float)

    WK_core = np.zeros_like(xM_vals, dtype=float)
    WP_core = np.zeros_like(xM_vals, dtype=float)

    if lmax_core is None:
        lmax_core, _ = get_lmax_core_shell(
            xM_vals,
            delta,
            c0, c1, c2,
            pad_core=pad_core,
            pad_shell=0
        )

    if lmax_cap is not None:
        lmax_core = min(lmax_core, int(lmax_cap))

    for ia, xM in enumerate(xM_vals):

        xA = (c0 / c1) * xM
        xB = (c0 / c2) * xM
        yM = (1.0 + delta) * xM
        yB = (1.0 + delta) * xB

        WKc = 0.0
        WPc = 0.0

        for l in range(lmax_core + 1):

            bl, cl, dl = coeffs(
                l,
                xA, xB, yM, yB,
                rho0, rho1, rho2,
                c0, c1, c2
            )

            bl2 = np.abs(bl)**2

            FA_l = FA(l, xA)
            BA_l = BA(l, xA)

            WPc += bl2 * (2*l + 1) * FA_l
            WKc += bl2 * (2*l + 1) * (FA_l + BA_l)

        WP_core[ia] = np.real_if_close(WPc)
        WK_core[ia] = np.real_if_close(WKc)

    return WK_core, WP_core


def WT_core_vs_xM(xM_vals, delta, rho0, rho1, rho2, c0, c1, c2, lmax_core=None, pad_core=0, lmax_cap=None):

    WK_core, WP_core = W_core_parts_vs_xM(
        xM_vals,
        delta,
        rho0, rho1, rho2,
        c0, c1, c2,
        lmax_core=lmax_core,
        pad_core=pad_core,
        lmax_cap=lmax_cap)

    return WK_core + WP_core

### Shell energy

For the shell, the potential energy is written as

\begin{equation}
\frac{2f_0W_P^{(B)}}{Z_B|A|^2}
=
\sum_{\ell=0}^{\infty}
(2\ell+1)
\left[
\frac{|c_\ell|^2}{2}F_{jj}(x_B,y_B)
+
\frac{|d_\ell|^2}{2}F_{yy}(x_B,y_B)
+
\mathrm{Re}(c_\ell^*d_\ell)F_{jy}(x_B,y_B)
\right],
\end{equation}

where

\begin{equation}
F_{jj}(x_B,y_B)
=
\left[
x^3
\left(
j_\ell^2(x)
-
j_{\ell-1}(x)j_{\ell+1}(x)
\right)
\right]_{x_B}^{y_B},
\end{equation}

\begin{equation}
F_{yy}(x_B,y_B)
=
\left[
x^3
\left(
y_\ell^2(x)
-
y_{\ell-1}(x)y_{\ell+1}(x)
\right)
\right]_{x_B}^{y_B},
\end{equation}

and

\begin{equation}
F_{jy}(x_B,y_B)
=
\left[
\frac{x^3}{2}
\left(
2j_\ell(x)y_\ell(x)
-
j_{\ell+1}(x)y_{\ell-1}(x)
-
j_{\ell-1}(x)y_{\ell+1}(x)
\right)
\right]_{x_B}^{y_B}.
\end{equation}

Using the closed form obtained after the cancellation between the radial and
angular contributions, the kinetic energy can be evaluated directly as

\begin{equation}
\begin{split}
\frac{2f_0W_K^{(B)}}{Z_B|A|^2}
=
\frac{2f_0W_P^{(B)}}{Z_B|A|^2}
+
\sum_{\ell=0}^{\infty}
(2\ell+1)
\Bigg\{
&
|c_\ell|^2
\left[
x^2j_\ell(x)j_\ell'(x)
\right]_{x_B}^{y_B}
\\
&
+
|d_\ell|^2
\left[
x^2y_\ell(x)y_\ell'(x)
\right]_{x_B}^{y_B}
\\
&
+
2\mathrm{Re}(c_\ell^*d_\ell)
\left[
x^2j_\ell(x)y_\ell'(x)
\right]_{x_B}^{y_B}
\Bigg\}.
\end{split}
\end{equation}

Thus, the total energy stored in the shell is

\begin{equation}
W_T^{(B)} = W_P^{(B)} + W_K^{(B)}.
\end{equation}

This form avoids the explicit numerical evaluation of the auxiliary integrals
\(I_{j,\ell}\), \(I_{y,\ell}\), and \(I_{jy,\ell}\), since these terms cancel when
the radial and angular kinetic contributions are combined.

In [ ]:
def W_shell_parts_vs_xM(
    xM_vals,
    delta,
    rho0, rho1, rho2,
    c0, c1, c2,
    lmax_shell=None,
    pad_shell=0,
    lmax_cap=None):


    xM_vals = np.asarray(xM_vals, dtype=float)

    WK_shell = np.zeros_like(xM_vals, dtype=float)
    WP_shell = np.zeros_like(xM_vals, dtype=float)

    if lmax_shell is None:
        _, lmax_shell = get_lmax_core_shell(
            xM_vals,
            delta,
            c0, c1, c2,
            pad_core=0,
            pad_shell=pad_shell
        )

    if lmax_cap is not None:
        lmax_shell = min(lmax_shell, int(lmax_cap))

    for ia, xM in enumerate(xM_vals):

        xA = (c0 / c1) * xM
        xB = (c0 / c2) * xM
        yM = (1.0 + delta) * xM
        yB = (1.0 + delta) * xB

        WKs = 0.0
        WPs = 0.0

        for l in range(lmax_shell + 1):

            bl, cl, dl = coeffs(l, xA, xB, yM, yB, rho0, rho1, rho2, c0, c1, c2)

            cl2 = np.abs(cl)**2
            dl2 = np.abs(dl)**2
            Re_cd = np.real(np.conj(cl) * dl)

            termo_P = (0.5 * cl2 * Fjj(l, xB, yB) + 0.5 * dl2 * Fyy(l, xB, yB) + Re_cd * Fjy(l, xB, yB))

            termo_B = (cl2 * Bjj(l, xB, yB) + dl2 * Byy(l, xB, yB) + 2.0 * Re_cd * Bjy(l, xB, yB))

            WPs += (2*l + 1) * termo_P
            WKs += (2*l + 1) * (termo_P + termo_B)

        WP_shell[ia] = np.real_if_close(WPs)
        WK_shell[ia] = np.real_if_close(WKs)

    return WK_shell, WP_shell


def WT_shell_vs_xM(xM_vals, delta, rho0, rho1, rho2, c0, c1, c2, lmax_shell=None, pad_shell=0, lmax_cap=None):

    WK_shell, WP_shell = W_shell_parts_vs_xM(xM_vals, delta, rho0, rho1, rho2, c0, c1, c2, lmax_shell=lmax_shell, pad_shell=pad_shell, lmax_cap=lmax_cap)

    return WK_shell + WP_shell

In [ ]:
import os
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap,Normalize
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from scipy.spatial import cKDTree

BASE_DIR=DADOS

CASES={
    "velocity":{
        "f":0.00,
        "checkpoint":os.path.join(
            BASE_DIR,
            "coreshell_lmax10_mixed_mix0.00",
            "checkpoint_coreshell_lmax10_mixed_mix0.00.npz"
        ),
        "xlabel":r"$\alpha_v=c_A/c_M$",
        "ylabel":r"$\beta_v=c_B/c_M$"
    },

    "mixed":{
        "f":0.50,
        "checkpoint":os.path.join(
            BASE_DIR,
            "coreshell_lmax10_mixed_mix0.50",
            "checkpoint_coreshell_lmax10_mixed_mix0.50.npz"
        ),
        "xlabel":r"$\alpha=Z_A/Z_M$",
        "ylabel":r"$\beta=Z_B/Z_M$"
    },

    "density":{
        "f":1.00,
        "checkpoint":os.path.join(
            BASE_DIR,
            "coreshell_lmax10_mixed_mix1.00",
            "checkpoint_coreshell_lmax10_mixed_mix1.00.npz"
        ),
        "xlabel":r"$\alpha_d=\rho_A/\rho_M$",
        "ylabel":r"$\beta_d=\rho_B/\rho_M$"
    }
}

if BASE_DIR.startswith("/content/drive"):
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        pass

OUT_DIR=os.path.join(
    BASE_DIR,
    "esquema_analise_resultados_WT_log"
)

os.makedirs(
    OUT_DIR,
    exist_ok=True
)

FIXED_DELTA=1.50

OVERLAY_DELTAS=[
    0.01,
    0.50
]

OVERLAY_COLORS=[
    "#600B00",
    "black"
]

OVERLAY_LINESTYLES=[
    (0,(4.5,2.5)),
    (0,(5.0,2.2))
]

OVERLAY_LINEWIDTHS=[
    1.9,
    2.0
]

OVERLAY_ALPHAS=[
    0.92,
    0.82
]

BASE_FRONTIER_COLOR="#7FA6D9"
BASE_FRONTIER_LINESTYLE=(0,(5.0,2.2))
BASE_FRONTIER_LINEWIDTH=2.0
BASE_FRONTIER_ALPHA=0.95

BLACK_FRONTIER_DELTA=0.50
BLACK_FRONTIER_HALO_COLOR="white"
BLACK_FRONTIER_HALO_WIDTH=0
BLACK_FRONTIER_HALO_ALPHA=1.0

MIN_Z=1e-3
MAX_Z=1000.0
N_LOG_BINS=20

BOX_COLOR="#600B00"
BOX_LINEWIDTH=1.8
CBAR_BOX_LINEWIDTH=1.8

CELL_LINECOLOR="white"
CELL_LINEWIDTH=0.42

CENTER_LINE_COLOR="#AEB9C5"
CENTER_LINE_STYLE=":"
CENTER_LINE_WIDTH=1.35
CENTER_LINE_ALPHA=0.35

FIG_FACE_COLOR="white"
FIGSIZE=(10.5,10.5)

LABEL_FONTSIZE=23
TICK_FONTSIZE=15

CBAR_LABEL_FONTSIZE=20
CBAR_TICK_FONTSIZE=15

LEGEND_FONTSIZE=20.0

DELTA_BOX_FONTSIZE=24
NRES_BOX_FONTSIZE=24

CBAR_LABEL_COLOR="black"

LEGEND_Y_OFFSET_ABOVE_BETA1=1.00

LEGEND_HANDLE_LENGTH=3.0
LEGEND_HANDLE_TEXT_PAD=1.0
LEGEND_BORDER_PAD=0.20
LEGEND_LABEL_SPACING=0.45
LEGEND_ZORDER=100

DENSITY_LEGEND_RIGHT_POSITION=19.0

RESONANCE_CMAP=LinearSegmentedColormap.from_list(
    "resonance_count",
    [
        "white",
        "#D9E1EA",
        "#526F92",
        "black"
    ],
    N=256
)

RESONANCE_CMAP.set_bad(
    "#D9D9D9"
)


def nearest_index(value,array):

    array=np.asarray(
        array,
        dtype=float
    )

    return int(
        np.argmin(
            np.abs(
                array-float(value)
            )
        )
    )


def style_axes_and_cbar(ax,cbar=None):

    ax.tick_params(
        axis="both",
        labelsize=TICK_FONTSIZE,
        width=1.25,
        length=4.5,
        direction="out"
    )

    for spine in ax.spines.values():

        spine.set_visible(True)
        spine.set_edgecolor(BOX_COLOR)
        spine.set_linewidth(BOX_LINEWIDTH)

    if cbar is not None:

        cbar.ax.tick_params(
            labelsize=CBAR_TICK_FONTSIZE,
            width=1.25,
            length=4.5,
            direction="out"
        )

        for spine in cbar.ax.spines.values():

            spine.set_visible(True)
            spine.set_edgecolor(BOX_COLOR)
            spine.set_linewidth(
                CBAR_BOX_LINEWIDTH
            )


def curva_valida(item):

    if item.get("status")!="ok":
        return False

    rep=item.get(
        "reparo_velocity"
    )

    return not (
        isinstance(rep,dict)
        and not rep.get("ok",True)
    )


def load_checkpoint_vetorizado(filepath):

    if not os.path.exists(filepath):

        raise FileNotFoundError(
            f"File not found:\n{filepath}"
        )

    with np.load(
        filepath,
        allow_pickle=True
    ) as data:

        if "curves" not in data.files:

            raise KeyError(
                "This file does not look like a vectorized checkpoint "
                "(missing 'curves')."
            )

        curvas=data["curves"].item()

        delta_vals_saved=None

        for nome in (
            "delta_values",
            "delta_vals"
        ):

            if nome in data.files:

                delta_vals_saved=np.asarray(
                    data[nome],
                    dtype=float
                )

                break

    alphas=[]
    betas=[]
    deltas=[]
    n_peaks=[]

    for item in curvas.values():

        if not curva_valida(item):
            continue

        alpha=float(
            item.get(
                "alpha",
                np.nan
            )
        )

        beta=float(
            item.get(
                "beta",
                np.nan
            )
        )

        delta=float(
            item.get(
                "delta",
                np.nan
            )
        )

        if not (
            np.isfinite(alpha)
            and alpha>0.0
            and np.isfinite(beta)
            and beta>0.0
            and np.isfinite(delta)
        ):
            continue

        alphas.append(alpha)
        betas.append(beta)
        deltas.append(delta)

        n_peaks.append(
            len(
                item.get(
                    "ressonancias_malha",
                    []
                ) or []
            )
        )

    alphas=np.asarray(
        alphas,
        dtype=float
    )

    betas=np.asarray(
        betas,
        dtype=float
    )

    deltas=np.asarray(
        deltas,
        dtype=float
    )

    n_peaks=np.asarray(
        n_peaks,
        dtype=float
    )

    delta_vals=(
        delta_vals_saved
        if delta_vals_saved is not None
        else np.unique(
            np.round(
                deltas,
                6
            )
        )
    )

    return {
        "delta_vals":delta_vals,
        "alpha":alphas,
        "beta":betas,
        "delta":deltas,
        "n_peaks":n_peaks
    }


def bin_delta_in_alpha_beta_plane(
    db,
    delta_value,
    log_edges
):

    delta_vals=np.asarray(
        db["delta_vals"],
        dtype=float
    )

    idd=nearest_index(
        delta_value,
        delta_vals
    )

    real_delta=float(
        delta_vals[idd]
    )

    point_delta_index=np.array(
        [
            nearest_index(
                d,
                delta_vals
            )
            for d in db["delta"]
        ],
        dtype=int
    )

    mask_delta=(
        point_delta_index==idd
    )

    alpha=db["alpha"][mask_delta]
    beta=db["beta"][mask_delta]
    peaks=db["n_peaks"][mask_delta]

    min_z=10.0**log_edges[0]
    max_z=10.0**log_edges[-1]

    valid=(
        np.isfinite(alpha)
        & np.isfinite(beta)
        & np.isfinite(peaks)
        & (alpha>=min_z)
        & (alpha<=max_z)
        & (beta>=min_z)
        & (beta<=max_z)
    )

    n_bins=len(log_edges)-1

    count_map=np.zeros(
        (n_bins,n_bins),
        dtype=int
    )

    max_map=np.full(
        (n_bins,n_bins),
        -np.inf,
        dtype=float
    )

    if not np.any(valid):

        return {
            "delta_index":idd,
            "real_delta":real_delta,
            "max_map":np.full(
                (n_bins,n_bins),
                np.nan
            ),
            "count_map":count_map
        }

    alpha=alpha[valid]
    beta=beta[valid]
    peaks=peaks[valid]

    log_alpha=np.log10(alpha)
    log_beta=np.log10(beta)

    i_alpha=np.searchsorted(
        log_edges,
        log_alpha,
        side="right"
    )-1

    i_beta=np.searchsorted(
        log_edges,
        log_beta,
        side="right"
    )-1

    i_alpha=np.clip(
        i_alpha,
        0,
        n_bins-1
    )

    i_beta=np.clip(
        i_beta,
        0,
        n_bins-1
    )

    np.add.at(
        count_map,
        (i_alpha,i_beta),
        1
    )

    np.maximum.at(
        max_map,
        (i_alpha,i_beta),
        peaks
    )

    occupied=count_map>0
    empty=~occupied

    if (
        np.any(empty)
        and np.any(occupied)
    ):

        log_centers=(
            log_edges[:-1]
            +log_edges[1:]
        )/2.0

        grid_a,grid_b=np.meshgrid(
            log_centers,
            log_centers,
            indexing="ij"
        )

        empty_coords=np.column_stack(
            [
                grid_a[empty],
                grid_b[empty]
            ]
        )

        valid_points=np.column_stack(
            [
                log_alpha,
                log_beta
            ]
        )

        _,nearest_indices=cKDTree(
            valid_points
        ).query(
            empty_coords
        )

        max_map[empty]=peaks[
            nearest_indices
        ]

    return {
        "delta_index":idd,
        "real_delta":real_delta,
        "max_map":max_map,
        "count_map":count_map
    }


def build_resonance_map(max_map):

    resonance_map=np.full(
        max_map.shape,
        np.nan,
        dtype=float
    )

    valid=np.isfinite(
        max_map
    )

    resonance_map[valid]=np.maximum(
        max_map[valid],
        0.0
    )

    return resonance_map


def build_resonance_norm(
    resonance_map
):

    finite_values=resonance_map[
        np.isfinite(
            resonance_map
        )
    ]

    if finite_values.size==0:

        return (
            Normalize(
                vmin=0.0,
                vmax=1.0
            ),
            0
        )

    max_resonances=int(
        np.ceil(
            np.max(
                finite_values
            )
        )
    )

    vmax=max(
        max_resonances,
        1
    )

    norm=Normalize(
        vmin=0.0,
        vmax=np.log10(
            1.0+vmax
        )
    )

    return (
        norm,
        max_resonances
    )


def to_scale(values):

    values=np.asarray(
        values,
        dtype=float
    )

    return np.log10(
        1.0+values
    )


def build_resonance_ticks_log(
    max_resonances,
    contrast_in
):

    ticks=(
        [0]
        +[
            10**k
            for k in range(0,6)
            if 10**k<0.8*max_resonances
        ]
        +[max_resonances]
    )

    ticks=np.unique(
        np.asarray(
            ticks,
            dtype=int
        )
    )

    if contrast_in in (
        "velocity",
        "mixed"
    ):

        ticks=ticks[
            ticks!=1
        ]

    return ticks


def get_boundary_segments(mask):

    n_alpha,n_beta=mask.shape

    segments=[]

    for i_alpha in range(
        n_alpha
    ):

        for i_beta in range(
            n_beta
        ):

            if not mask[
                i_alpha,
                i_beta
            ]:
                continue

            x_left=i_alpha-0.5
            x_right=i_alpha+0.5

            y_bottom=i_beta-0.5
            y_top=i_beta+0.5

            if (
                i_alpha==0
                or not mask[
                    i_alpha-1,
                    i_beta
                ]
            ):

                segments.append(
                    (
                        (
                            x_left,
                            y_bottom
                        ),
                        (
                            x_left,
                            y_top
                        )
                    )
                )

            if (
                i_alpha==n_alpha-1
                or not mask[
                    i_alpha+1,
                    i_beta
                ]
            ):

                segments.append(
                    (
                        (
                            x_right,
                            y_bottom
                        ),
                        (
                            x_right,
                            y_top
                        )
                    )
                )

            if (
                i_beta==0
                or not mask[
                    i_alpha,
                    i_beta-1
                ]
            ):

                segments.append(
                    (
                        (
                            x_left,
                            y_bottom
                        ),
                        (
                            x_right,
                            y_bottom
                        )
                    )
                )

            if (
                i_beta==n_beta-1
                or not mask[
                    i_alpha,
                    i_beta+1
                ]
            ):

                segments.append(
                    (
                        (
                            x_left,
                            y_top
                        ),
                        (
                            x_right,
                            y_top
                        )
                    )
                )

    return segments


def draw_delta_frontiers(
    ax,
    base_stats,
    overlay_stats
):

    legend_handles=[]

    for start_pt,end_pt in get_boundary_segments(
        base_stats["max_map"]>0
    ):

        ax.plot(
            [
                start_pt[0],
                end_pt[0]
            ],
            [
                start_pt[1],
                end_pt[1]
            ],
            color=BASE_FRONTIER_COLOR,
            linewidth=BASE_FRONTIER_LINEWIDTH,
            linestyle=BASE_FRONTIER_LINESTYLE,
            alpha=BASE_FRONTIER_ALPHA,
            solid_capstyle="butt",
            zorder=17
        )

    legend_handles.append(
        Line2D(
            [0],
            [0],
            color=OVERLAY_COLORS[0],
            linewidth=OVERLAY_LINEWIDTHS[0],
            linestyle=OVERLAY_LINESTYLES[0],
            alpha=OVERLAY_ALPHAS[0],
            label=rf"Reference frontier: $\delta={overlay_stats[0]['real_delta']:g}$"
        )
    )

    legend_handles.append(
        Line2D(
            [0],
            [0],
            color=OVERLAY_COLORS[1],
            linewidth=OVERLAY_LINEWIDTHS[1],
            linestyle=OVERLAY_LINESTYLES[1],
            alpha=OVERLAY_ALPHAS[1],
            label=rf"Reference frontier: $\delta={overlay_stats[1]['real_delta']:g}$"
        )
    )

    legend_handles.append(
        Line2D(
            [0],
            [0],
            color=BASE_FRONTIER_COLOR,
            linewidth=BASE_FRONTIER_LINEWIDTH,
            linestyle=BASE_FRONTIER_LINESTYLE,
            alpha=BASE_FRONTIER_ALPHA,
            label=rf"Reference frontier: $\delta={base_stats['real_delta']:g}$"
        )
    )

    for i,stats in enumerate(
        overlay_stats
    ):

        color=OVERLAY_COLORS[i]
        linestyle=OVERLAY_LINESTYLES[i]
        linewidth=OVERLAY_LINEWIDTHS[i]
        alpha=OVERLAY_ALPHAS[i]

        segments=get_boundary_segments(
            stats["max_map"]>0
        )

        if not segments:
            continue

        is_black_frontier=np.isclose(
            stats["real_delta"],
            BLACK_FRONTIER_DELTA
        )

        for start_pt,end_pt in segments:

            x_values=[
                start_pt[0],
                end_pt[0]
            ]

            y_values=[
                start_pt[1],
                end_pt[1]
            ]

            if is_black_frontier:

                ax.plot(
                    x_values,
                    y_values,
                    color=BLACK_FRONTIER_HALO_COLOR,
                    linewidth=linewidth+BLACK_FRONTIER_HALO_WIDTH,
                    linestyle=linestyle,
                    alpha=BLACK_FRONTIER_HALO_ALPHA,
                    solid_capstyle="butt",
                    zorder=16
                )

            ax.plot(
                x_values,
                y_values,
                color=color,
                linewidth=linewidth,
                linestyle=linestyle,
                alpha=alpha,
                solid_capstyle="butt",
                zorder=18+i
            )

    return legend_handles


def draw_square_bins(
    ax,
    matrix,
    cmap,
    norm
):

    n_alpha,n_beta=matrix.shape

    bad_color=cmap.get_bad()

    for i_alpha in range(
        n_alpha
    ):

        for i_beta in range(
            n_beta
        ):

            value=matrix[
                i_alpha,
                i_beta
            ]

            if np.isfinite(value):

                facecolor=cmap(
                    norm(
                        to_scale(value)
                    )
                )

            else:

                facecolor=bad_color

            ax.add_patch(
                Rectangle(
                    (
                        i_alpha-0.5,
                        i_beta-0.5
                    ),
                    1.0,
                    1.0,
                    facecolor=facecolor,
                    edgecolor=CELL_LINECOLOR,
                    linewidth=CELL_LINEWIDTH,
                    zorder=1
                )
            )


def log_value_to_cell_coordinate(
    value,
    log_edges
):

    n_bins=len(log_edges)-1

    return (
        (
            np.log10(value)
            -log_edges[0]
        )
        /(
            log_edges[-1]
            -log_edges[0]
        )
        *n_bins
        -0.5
    )


def configure_alpha_beta_axes(
    ax,
    log_edges,
    xlabel,
    ylabel
):

    n_bins=len(log_edges)-1

    ax.plot(
        [
            -0.5,
            n_bins-0.5
        ],
        [
            -0.5,
            n_bins-0.5
        ],
        color=CENTER_LINE_COLOR,
        linestyle=CENTER_LINE_STYLE,
        linewidth=CENTER_LINE_WIDTH*1.8,
        alpha=CENTER_LINE_ALPHA,
        zorder=6
    )

    one_position=log_value_to_cell_coordinate(
        1.0,
        log_edges
    )

    ax.axvline(
        one_position,
        color=CENTER_LINE_COLOR,
        linestyle=CENTER_LINE_STYLE,
        linewidth=CENTER_LINE_WIDTH*1.8,
        alpha=CENTER_LINE_ALPHA,
        zorder=6
    )

    ax.axhline(
        one_position,
        color=CENTER_LINE_COLOR,
        linestyle=CENTER_LINE_STYLE,
        linewidth=CENTER_LINE_WIDTH*1.8,
        alpha=CENTER_LINE_ALPHA,
        zorder=6
    )

    candidate_ticks=np.array(
        [
            1e-3,
            1e-2,
            1e-1,
            1.0,
            10.0,
            100.0,
            1000.0
        ],
        dtype=float
    )

    min_value=10.0**log_edges[0]
    max_value=10.0**log_edges[-1]

    tick_values=candidate_ticks[
        (candidate_ticks>=min_value)
        & (candidate_ticks<=max_value)
    ]

    tick_positions=[
        log_value_to_cell_coordinate(
            value,
            log_edges
        )
        for value in tick_values
    ]

    tick_labels=[
        f"{value:g}"
        for value in tick_values
    ]

    ax.set_xticks(
        tick_positions
    )

    ax.set_xticklabels(
        tick_labels
    )

    ax.set_yticks(
        tick_positions
    )

    ax.set_yticklabels(
        tick_labels
    )

    ax.set_xlabel(
        xlabel,
        fontsize=LABEL_FONTSIZE,
        labelpad=9
    )

    ax.set_ylabel(
        ylabel,
        fontsize=LABEL_FONTSIZE,
        labelpad=10
    )

    ax.set_xlim(
        -0.5,
        n_bins-0.5
    )

    ax.set_ylim(
        -0.5,
        n_bins-0.5
    )

    ax.set_aspect(
        "equal",
        adjustable="box"
    )

    style_axes_and_cbar(
        ax
    )


def plot_za_zb_fixed_delta(
    db,
    fixed_delta,
    contrast_in,
    mix_factor,
    xlabel,
    ylabel,
    data_file
):

    log_edges=np.linspace(
        np.log10(MIN_Z),
        np.log10(MAX_Z),
        N_LOG_BINS+1
    )

    base_stats=bin_delta_in_alpha_beta_plane(
        db=db,
        delta_value=fixed_delta,
        log_edges=log_edges
    )

    overlay_stats=[
        bin_delta_in_alpha_beta_plane(
            db=db,
            delta_value=delta,
            log_edges=log_edges
        )
        for delta in OVERLAY_DELTAS
    ]

    resonance_map=build_resonance_map(
        base_stats["max_map"]
    )

    resonance_norm,max_resonances=build_resonance_norm(
        resonance_map
    )

    resonance_ticks=build_resonance_ticks_log(
        max_resonances,
        contrast_in
    )

    fig=plt.figure(
        figsize=FIGSIZE
    )

    fig.patch.set_facecolor(
        FIG_FACE_COLOR
    )

    ax_class=fig.add_axes(
        [
            0.12,
            0.13,
            0.70,
            0.70
        ]
    )

    cax_class=fig.add_axes(
        [
            0.84,
            0.13,
            0.03,
            0.70
        ]
    )

    draw_square_bins(
        ax=ax_class,
        matrix=resonance_map,
        cmap=RESONANCE_CMAP,
        norm=resonance_norm
    )

    configure_alpha_beta_axes(
        ax=ax_class,
        log_edges=log_edges,
        xlabel=xlabel,
        ylabel=ylabel
    )

    frontier_handles=draw_delta_frontiers(
        ax=ax_class,
        base_stats=base_stats,
        overlay_stats=overlay_stats
    )

    if frontier_handles:

        one_position=log_value_to_cell_coordinate(
            1.0,
            log_edges
        )

        if contrast_in=="density":

            legend=ax_class.legend(
                handles=frontier_handles,

                loc="center right",

                bbox_to_anchor=(
                    DENSITY_LEGEND_RIGHT_POSITION,
                    one_position
                ),

                bbox_transform=ax_class.transData,

                fontsize=LEGEND_FONTSIZE,

                frameon=True,
                framealpha=1.0,
                facecolor="white",
                edgecolor="none",
                fancybox=False,

                borderpad=LEGEND_BORDER_PAD,
                handlelength=LEGEND_HANDLE_LENGTH,
                handletextpad=LEGEND_HANDLE_TEXT_PAD,
                labelspacing=LEGEND_LABEL_SPACING,
                borderaxespad=0.0
            )

        else:

            legend=ax_class.legend(
                handles=frontier_handles,

                loc="lower right",

                bbox_to_anchor=(
                    19.15,
                    one_position+LEGEND_Y_OFFSET_ABOVE_BETA1
                ),

                bbox_transform=ax_class.transData,

                fontsize=LEGEND_FONTSIZE,

                frameon=True,
                framealpha=1.0,
                facecolor="white",
                edgecolor="none",
                fancybox=False,

                borderpad=LEGEND_BORDER_PAD,
                handlelength=LEGEND_HANDLE_LENGTH,
                handletextpad=LEGEND_HANDLE_TEXT_PAD,
                labelspacing=LEGEND_LABEL_SPACING,
                borderaxespad=0.0
            )

        legend.set_zorder(
            LEGEND_ZORDER
        )

        legend.get_frame().set_facecolor(
            "white"
        )

        legend.get_frame().set_alpha(
            1.0
        )

        legend.get_frame().set_edgecolor(
            "none"
        )

        legend.get_frame().set_linewidth(
            0.0
        )

    ax_class.text(
        0.50,
        0.965,
        rf"$\delta={base_stats['real_delta']:g}$",
        transform=ax_class.transAxes,
        fontsize=DELTA_BOX_FONTSIZE,
        color="white",
        va="top",
        ha="center",
        bbox=dict(
            facecolor=BOX_COLOR,
            edgecolor="none",
            alpha=1.0,
            boxstyle="square,pad=0.16"
        ),
        zorder=110
    )

    ax_class.text(
        0.955,
        0.965,
        r"$N_{\mathrm{res}}$",
        transform=ax_class.transAxes,
        fontsize=NRES_BOX_FONTSIZE,
        color="white",
        va="top",
        ha="right",
        bbox=dict(
            facecolor=BOX_COLOR,
            edgecolor="none",
            alpha=1.0,
            boxstyle="square,pad=0.14"
        ),
        zorder=111
    )

    resonance_mapper=cm.ScalarMappable(
        norm=resonance_norm,
        cmap=RESONANCE_CMAP
    )

    resonance_mapper.set_array([])

    cbar_class=fig.colorbar(
        resonance_mapper,
        cax=cax_class,
        ticks=to_scale(
            resonance_ticks
        )
    )

    cbar_class.ax.set_yticklabels(
        [
            str(value)
            for value in resonance_ticks
        ]
    )

    cbar_class.set_label(
        "Resonances",
        fontsize=CBAR_LABEL_FONTSIZE,
        labelpad=18,
        rotation=270,
        color=CBAR_LABEL_COLOR
    )

    style_axes_and_cbar(
        ax_class,
        cbar_class
    )

    output_base=os.path.join(
        OUT_DIR,
        f"ZA_ZB_{contrast_in}_f{mix_factor:.2f}"
        f"_delta_{base_stats['real_delta']:g}"
        "_WT_peak_count_log_filled"
        "_delta001_delta050"
        "_reference_frontiers_halo"
    )

    fig.savefig(
        output_base+".png",
        dpi=600,
        bbox_inches="tight"
    )

    fig.savefig(
        output_base+".pdf",
        dpi=600,
        bbox_inches="tight"
    )

    fig.savefig(
        output_base+".eps",
        dpi=600,
        bbox_inches="tight"
    )

    np.savez_compressed(
        output_base+"_dados.npz",

        data_file=data_file,

        contrast_in=contrast_in,

        mix_factor=mix_factor,

        fixed_delta=base_stats[
            "real_delta"
        ],

        overlay_deltas=np.asarray(
            [
                stats["real_delta"]
                for stats in overlay_stats
            ],
            dtype=float
        ),

        min_z=MIN_Z,
        max_z=MAX_Z,

        n_log_bins=N_LOG_BINS,

        log_edges=log_edges,

        max_map=base_stats[
            "max_map"
        ],

        samples_per_bin=base_stats[
            "count_map"
        ],

        resonance_map=resonance_map,

        resonance_scale_max=max_resonances,

        escala="log",

        resonance_ticks=resonance_ticks
    )

    plt.show()

    plt.close(
        fig
    )

    return (
        base_stats,
        resonance_map
    )


if __name__=="__main__":

    print(
        "Generating fixed-delta ZA x ZB WT-peak maps..."
    )

    for contrast_in,case in CASES.items():

        checkpoint_path=case[
            "checkpoint"
        ]

        print(
            "\n"
            +"="*80
        )

        print(
            f"Contrast: {contrast_in}"
        )

        print(
            f"Mix factor: {case['f']}"
        )

        print(
            f"Checkpoint: {checkpoint_path}"
        )

        try:

            db=load_checkpoint_vetorizado(
                checkpoint_path
            )

            print(
                f"Curves in checkpoint: "
                f"{len(db['alpha'])}"
            )

            base_stats,resonance_map=plot_za_zb_fixed_delta(
                db=db,
                fixed_delta=FIXED_DELTA,
                contrast_in=contrast_in,
                mix_factor=case["f"],
                xlabel=case["xlabel"],
                ylabel=case["ylabel"],
                data_file=checkpoint_path
            )

            occupied=(
                base_stats[
                    "count_map"
                ]>0
            )

            print(
                "\nGenerated-map summary:"
            )

            print(
                "Background delta:",
                f"{base_stats['real_delta']:g}"
            )

            finite_resonances=resonance_map[
                np.isfinite(
                    resonance_map
                )
            ]

            if finite_resonances.size:

                print(
                    "Resonance scale:",
                    "0 to",
                    f"{int(np.ceil(np.max(finite_resonances)))}"
                )

            if np.any(
                occupied
            ):

                print(
                    "Original points per cell:",
                    f"{int(np.min(base_stats['count_map'][occupied]))}",
                    "to",
                    f"{int(np.max(base_stats['count_map'][occupied]))}"
                )

        except FileNotFoundError:

            print(
                f"File not found:\n"
                f"{checkpoint_path}"
            )

        except (
            KeyError,
            ValueError,
            RuntimeError
        ) as error:

            print(
                f"Error while generating the map:\n"
                f"{error}"
            )

    print(
        "\nFiles saved in:"
    )

    print(
        OUT_DIR
    )

In [ ]:
import os,json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator,NullFormatter
from scipy.optimize import minimize_scalar
from scipy.signal import find_peaks
from scipy.special import spherical_jn,spherical_yn
LMAX=10
X_MIN=1e-4
X_MAX=1.0
N_LOG=1000
N_MEIO=3000
N_FIM=1000
PROM_WT=0.1
PROM_WL=0.75
Q_MIN=2.0
Q_MIN_POLO=2.0
DX_MIN=1e-9
N_MAX_PONTOS=2_000_000
RHO_AGUA=1000.0
C_AGUA=1480.0
NORMALIZACAO="ZM"

BASE_DIR=DADOS

CASES={
    "velocity":{
        "f":0.00,
        "folder":os.path.join(BASE_DIR,"coreshell_lmax10_mixed_mix0.00"),
        "checkpoint":os.path.join(BASE_DIR,"coreshell_lmax10_mixed_mix0.00","checkpoint_coreshell_lmax10_mixed_mix0.00.npz"),
        "param":os.path.join(BASE_DIR,"coreshell_lmax10_mixed_mix0.00","parametros_coreshell_lmax10_mixed_mix0.00.json")
    },
    "mixed":{
        "f":0.50,
        "folder":os.path.join(BASE_DIR,"coreshell_lmax10_mixed_mix0.50"),
        "checkpoint":os.path.join(BASE_DIR,"coreshell_lmax10_mixed_mix0.50","checkpoint_coreshell_lmax10_mixed_mix0.50.npz"),
        "param":os.path.join(BASE_DIR,"coreshell_lmax10_mixed_mix0.50","parametros_coreshell_lmax10_mixed_mix0.50.json")
    },
    "density":{
        "f":1.00,
        "folder":os.path.join(BASE_DIR,"coreshell_lmax10_mixed_mix1.00"),
        "checkpoint":os.path.join(BASE_DIR,"coreshell_lmax10_mixed_mix1.00","checkpoint_coreshell_lmax10_mixed_mix1.00.npz"),
        "param":os.path.join(BASE_DIR,"coreshell_lmax10_mixed_mix1.00","parametros_coreshell_lmax10_mixed_mix1.00.json")
    }
}

OUT_DIR=os.path.join(BASE_DIR,"graficos_WT_mesma_malha_gerador")
os.makedirs(OUT_DIR,exist_ok=True)

def J(n,x):
    if n>=0:return spherical_jn(n,x)
    m=-n-1
    return (-1)**(m+1)*spherical_yn(m,x)

def Y(n,x):
    if n>=0:return spherical_yn(n,x)
    m=-n-1
    return (-1)**m*spherical_jn(m,x)

def dJ(n,x):
    if n>=0:return spherical_jn(n,x,derivative=True)
    m=-n-1
    return (-1)**(m+1)*spherical_yn(m,x,derivative=True)

def dY(n,x):
    if n>=0:return spherical_yn(n,x,derivative=True)
    m=-n-1
    return (-1)**m*spherical_jn(m,x,derivative=True)

def H(n,x):
    if n>=0:return spherical_jn(n,x)+1j*spherical_yn(n,x)
    m=-n-1
    return 1j*(-1)**m*(spherical_jn(m,x)+1j*spherical_yn(m,x))

def dH(n,x):
    if n>=0:return spherical_jn(n,x,derivative=True)+1j*spherical_yn(n,x,derivative=True)
    m=-n-1
    return 1j*(-1)**m*(spherical_jn(m,x,derivative=True)+1j*spherical_yn(m,x,derivative=True))

def Fjj(l,xB,yB):
    F=lambda x:x**3*(J(l,x)**2-J(l-1,x)*J(l+1,x))
    return F(yB)-F(xB)

def Fyy(l,xB,yB):
    F=lambda x:x**3*(Y(l,x)**2-Y(l-1,x)*Y(l+1,x))
    return F(yB)-F(xB)

def Fjy(l,xB,yB):
    F=lambda x:0.5*x**3*(2*J(l,x)*Y(l,x)-J(l+1,x)*Y(l-1,x)-J(l-1,x)*Y(l+1,x))
    return F(yB)-F(xB)

def Bjj(l,xB,yB):
    B=lambda x:x**2*J(l,x)*dJ(l,x)
    return B(yB)-B(xB)

def Byy(l,xB,yB):
    B=lambda x:x**2*Y(l,x)*dY(l,x)
    return B(yB)-B(xB)

def Bjy(l,xB,yB):
    B=lambda x:x**2*J(l,x)*dY(l,x)
    return B(yB)-B(xB)

def FA(l,xA):
    return 0.5*xA**3*(J(l,xA)**2-J(l-1,xA)*J(l+1,xA))

def BA(l,xA):
    return xA**2*J(l,xA)*dJ(l,xA)

def blocos(l,xA,xB,yM,yB,rho0,rho1,rho2,c0,c1,c2):
    rho_MB=rho0/rho2
    rho_AB=rho1/rho2
    k_MB=c2/c0
    k_AB=c2/c1
    A=-J(l,yB)
    B=-Y(l,yB)
    F=rho_MB*H(l,yM)
    G=-dJ(l,yB)
    HH=-dY(l,yB)
    JJ=k_MB*dH(l,yM)
    L=-rho_AB*J(l,xA)
    M=J(l,xB)
    P=Y(l,xB)
    Q=-k_AB*dJ(l,xA)
    R=dJ(l,xB)
    S=dY(l,xB)
    D=(L*R-M*Q)*(F*HH-B*JJ)+(L*S-P*Q)*(A*JJ-F*G)
    return D,L,M,P,Q,R,S,rho_MB,k_MB

def coeffs(l,xA,xB,yM,yB,*mat):
    D,L,M,P,Q,R,S,rho_MB,k_MB=blocos(l,xA,xB,yM,yB,*mat)
    X=1j*rho_MB*k_MB/yM**2
    return (M*S-P*R)*X/D,(P*Q-L*S)*X/D,(L*R-M*Q)*X/D

def argumentos(z,delta,mat):
    rho0,rho1,rho2,c0,c1,c2=mat
    xB=(c0/c2)*z
    return (c0/c1)*z,xB,(1+delta)*z,(1+delta)*xB

def pesos_Z(mat):
    if NORMALIZACAO=="separada":
        return 1.0,1.0
    rho0,rho1,rho2,c0,c1,c2=mat
    ZM=rho0*c0
    return rho1*c1/ZM,rho2*c2/ZM

def energia_l(l,x,delta,mat):
    x=np.asarray(x,float)
    xA,xB,yM,yB=argumentos(x,delta,mat)
    f=2*l+1
    with np.errstate(all="ignore"):
        bl,cl,dl=coeffs(l,xA,xB,yM,yB,*mat)
        nucleo=np.real(np.abs(bl)**2*f*(2*FA(l,xA)+BA(l,xA)))
        cl2=np.abs(cl)**2
        dl2=np.abs(dl)**2
        cd=np.real(np.conj(cl)*dl)
        pot=0.5*cl2*Fjj(l,xB,yB)+0.5*dl2*Fyy(l,xB,yB)+cd*Fjy(l,xB,yB)
        borda=cl2*Bjj(l,xB,yB)+dl2*Byy(l,xB,yB)+2*cd*Bjy(l,xB,yB)
        wA,wB=pesos_Z(mat)
        w=wA*nucleo+wB*np.real(f*(2*pot+borda))
    return np.where(np.isfinite(w),w,np.nan)

def energia_total(x,delta,mat,lmax=LMAX,bloco=400000):
    x=np.asarray(x,float)
    if len(x)>bloco:
        return np.concatenate([
            energia_total(x[i:i+bloco],delta,mat,lmax)
            for i in range(0,len(x),bloco)
        ])
    W=np.zeros_like(x)
    for l in range(lmax+1):
        W+=np.nan_to_num(energia_l(l,x,delta,mat))
    return np.where(np.isfinite(W)&(W>0),W,0.0)

def D_l(l,z,delta,mat):
    z=np.asarray(z,dtype=complex)
    with np.errstate(all="ignore"):
        return z**4*blocos(l,*argumentos(z,delta,mat),*mat)[0]

def materiais(alpha,beta,f):
    ZM=RHO_AGUA*C_AGUA
    rho0=RHO_AGUA
    rho1=RHO_AGUA*alpha**f
    rho2=RHO_AGUA*beta**f
    c0=ZM/RHO_AGUA
    c1=C_AGUA*alpha**(1-f)
    c2=C_AGUA*beta**(1-f)
    return rho0,rho1,rho2,c0,c1,c2

def eta_from_alpha_beta(alpha,beta):
    return (float(alpha)-float(beta))/(float(alpha)+float(beta))

def malha_base(delta,mat,x_min=X_MIN,x_max=X_MAX):
    rho0,rho1,rho2,c0,c1,c2=mat
    esp=np.pi/(c0/c1+delta*c0/c2)
    ext=min(esp,0.05*x_max)
    a=min(0.1,x_max)
    b=min(0.9,x_max)
    x=np.unique(np.concatenate([
        np.geomspace(x_min,a,N_LOG),
        np.linspace(a,b,N_MEIO),
        np.linspace(b,x_max+ext,N_FIM)
    ]))
    return x,esp

def maximos(energia,x,esp,prom=None,x_min=X_MIN,x_max=X_MAX,w=None):
    w=energia(x) if w is None else w

    if prom is None:
        idx=find_peaks(np.where(np.isfinite(w),w,-np.inf))[0]
    else:
        ok=np.isfinite(w)&(w>0)

        if not ok.any():
            return np.array([],int),np.array([])

        piso=max(w[ok].min()*1e-14,np.finfo(float).tiny)

        idx=find_peaks(
            np.log10(
                np.where(
                    ok,
                    np.maximum(w,piso),
                    piso
                )
            ),
            prominence=prom
        )[0]

    fica=[]
    xs=[]

    for i in idx:
        xp=float(x[i])

        if xp>x_max-2*min(esp,0.05*x_max) and 0<i<len(x)-1:
            alvo=lambda t:-np.log(
                max(
                    float(energia(np.array([t]))[0]),
                    np.finfo(float).tiny
                )
            )

            xp=float(
                minimize_scalar(
                    alvo,
                    bounds=(x[i-1],x[i+1]),
                    method="bounded",
                    options={"xatol":1e-12}
                ).x
            )

        if x_min<=xp<=x_max:
            fica.append(i)
            xs.append(xp)

    return np.asarray(fica,int),np.asarray(xs,float)

def refina_posicoes(energia,x,xs,it=60):
    if len(xs)==0:
        return xs

    i=np.clip(
        np.searchsorted(x,xs),
        1,
        len(x)-1
    )

    a=x[i-1]
    b=x[np.minimum(i+1,len(x)-1)]
    g=(np.sqrt(5)-1)/2
    c=b-g*(b-a)
    d=a+g*(b-a)
    fc,fd=energia(c),energia(d)

    for _ in range(it):
        m=np.nan_to_num(fc,nan=-np.inf)>np.nan_to_num(fd,nan=-np.inf)
        b,a=np.where(m,d,b),np.where(m,a,c)
        c,d=b-g*(b-a),a+g*(b-a)
        fc,fd=energia(c),energia(d)

    return 0.5*(a+b)

def casar(xs,zeros,esp):
    pares=sorted(
        (abs(x-z.real),i,j)
        for i,x in enumerate(xs)
        for j,z in enumerate(zeros)
        if abs(x-z.real)<=max(2*abs(z.imag),0.25*esp)
    )

    usado_x=set()
    usado_z=set()
    casados={}

    for _,i,j in pares:
        if i not in usado_x and j not in usado_z:
            usado_x.add(i)
            usado_z.add(j)
            casados[j]=i

    return casados

def modos_por_l(delta,mat,zeros_por_l,x,esp,lmax=LMAX):
    saida=[]

    for l in range(lmax+1):
        zeros=zeros_por_l.get(l)
        E=lambda v,l=l:energia_l(l,v,delta,mat)
        _,xs=maximos(E,x,esp)

        r={
            "l":l,
            "N_zeros":None if zeros is None else len(zeros),
            "N_max_malha":len(xs)
        }

        if zeros is not None:
            cas=casar(xs,zeros,esp)

            if len(xs)==len(zeros)==len(cas):
                r.update(
                    regra="bate",
                    x=[z.real for z in zeros]
                )
                saida.append(r)
                continue

            bons=[
                j
                for j,z in enumerate(zeros)
                if z.real/(2*abs(z.imag))>=Q_MIN
            ]

            sobra_x=[
                v
                for i,v in enumerate(xs)
                if i not in {
                    cas[j]
                    for j in cas
                    if j not in bons
                }
            ]

            z2=[
                zeros[j]
                for j in bons
            ]

            if len(sobra_x)==len(z2)==len(casar(sobra_x,z2,esp)):
                r.update(
                    regra="bate_Q2",
                    x=[z.real for z in z2]
                )
                saida.append(r)
                continue

        _,xp=maximos(
            E,
            x,
            esp,
            prom=PROM_WL
        )

        r.update(
            regra="proeminencia" if zeros is not None else "sem_polos",
            x=list(
                refina_posicoes(
                    E,
                    x,
                    xp
                )
            )
        )

        saida.append(r)

    return saida

def separacao_laurent(polos,delta,mat,n=400):
    if len(polos)<2:
        return np.array([]),np.array([])

    pol=sorted(
        polos,
        key=lambda p:p["pole_real"]
    )

    z=np.array([
        complex(
            p["pole_real"],
            p["pole_imag"]
        )
        for p in pol
    ])

    l=np.array([
        p["l"]
        for p in pol
    ])

    A=np.empty(len(z),complex)
    h=np.empty(len(z))

    for ll in np.unique(l):
        k=l==ll
        zz=z[k]
        e=1e-7*np.maximum(np.abs(zz),1e-4)

        A[k]=2*e/(
            D_l(ll,zz+e,delta,mat)
            -D_l(ll,zz-e,delta,mat)
        )

        gam=np.abs(zz.imag)
        s_=np.maximum(
            gam,
            1e-6*zz.real
        )

        h[k]=(
            energia_l(
                ll,
                zz.real+s_,
                delta,
                mat
            )
            *(s_**2+gam**2)
            /gam**2
        )

    z1,z2=z[:-1],z[1:]
    x1,x2=z1.real,z2.real
    G1=np.abs(z1.imag)[:,None]
    G2=np.abs(z2.imag)[:,None]

    g1=np.maximum(
        G1[:,0],
        1e3*np.finfo(float).eps*x1
    )

    g2=np.maximum(
        G2[:,0],
        1e3*np.finfo(float).eps*x2
    )

    u=np.geomspace(
        1e-3,
        30,
        n//4
    )

    X=np.sort(
        np.concatenate([
            x1[:,None]-g1[:,None]*u[::-1],
            x1[:,None]+g1[:,None]*u,
            np.linspace(x1,x2,n//4+2,axis=1)[:,1:-1],
            x2[:,None]-g2[:,None]*u[::-1],
            x2[:,None]+g2[:,None]*u
        ],axis=1),
        axis=1
    )

    mesmo=(l[:-1]==l[1:])[:,None]

    with np.errstate(all="ignore"):
        W=np.where(
            mesmo,
            np.abs(
                A[:-1,None]/(X-z1[:,None])
                +A[1:,None]/(X-z2[:,None])
            )**2,
            h[:-1,None]*G1**2/(
                (X-x1[:,None])**2+G1**2
            )
            +h[1:,None]*G2**2/(
                (X-x2[:,None])**2+G2**2
            )
        )

    mx=(
        (W[:,1:-1]>W[:,:-2])
        &(W[:,1:-1]>=W[:,2:])
        &np.isfinite(W[:,1:-1])
    )

    sep=np.full(
        len(z1),
        np.nan
    )

    for i in np.flatnonzero(
        mx.sum(axis=1)>=2
    ):
        p=X[i,1:-1][mx[i]]
        sep[i]=p.max()-p.min()

    return 0.5*(x1+x2),sep

def malha_dx(x,alvos,esp,laurent=None,x_max=X_MAX):
    A=sorted(
        [
            a
            for a in alvos
            if np.isfinite(a[1])
            and a[1]>0
        ],
        key=lambda a:a[0]
    )

    if not A:
        return x,np.nan

    P=np.array([
        a[0]
        for a in A
    ])

    Dn=np.array([
        a[1]
        for a in A
    ])

    ext=min(
        esp,
        0.05*x_max
    )

    seg=[]

    for gi in np.split(
        np.arange(len(P)),
        np.flatnonzero(
            np.diff(P)>0.5*esp
        )+1
    ):
        lo=max(
            P[gi[0]]-0.25*esp,
            X_MIN
        )

        hi=min(
            P[gi[-1]]+0.25*esp,
            x_max+ext
        )

        d=float(
            Dn[gi].min()
        )

        if laurent is not None and len(laurent[0]):
            k=(
                (laurent[0]>=lo)
                &(laurent[0]<=hi)
                &np.isfinite(laurent[1])
                &(laurent[1]>0)
            )

            if k.any():
                d=min(
                    d,
                    0.5*float(
                        laurent[1][k].min()
                    )
                )

        seg.append(
            (lo,hi,d,gi)
        )

    livre=max(
        N_MAX_PONTOS-len(x),
        1000
    )

    custo=lambda piso:sum(
        (hi-lo)/max(d,piso)
        for lo,hi,d,_ in seg
    )

    piso=DX_MIN

    if custo(piso)>livre:
        a=DX_MIN

        b=max(
            hi-lo
            for lo,hi,_,_ in seg
        )

        for _ in range(60):
            m=np.sqrt(a*b)

            a,b=(
                (m,b)
                if custo(m)>livre
                else (a,m)
            )

        piso=b

    extras=[]

    for lo,hi,d,gi in seg:
        h=max(
            d,
            piso
        )

        extras.append(
            lo+h*np.arange(
                int((hi-lo)/h)+1
            )
        )

    return (
        np.unique(
            np.concatenate(
                [x]+extras
            )
        ),
        float(piso)
    )

def ler_normalizacao(param_file):
    if not os.path.exists(param_file):
        return "ZM"

    try:
        with open(
            param_file,
            "r",
            encoding="utf-8"
        ) as f:
            data=json.load(f)

        args=data.get("argumentos") or {}

        if args.get("normalizacao") in (
            "ZM",
            "separada"
        ):
            return args[
                "normalizacao"
            ]

        const=data.get(
            "constantes"
        ) or {}

        if const.get(
            "NORMALIZACAO"
        ) in (
            "ZM",
            "separada"
        ):
            return const[
                "NORMALIZACAO"
            ]

    except Exception:
        pass

    return "ZM"

def reconstruir_malha(item,f):
    alpha=float(item["alpha"])
    beta=float(item["beta"])
    delta=float(item["delta"])

    mat=materiais(
        alpha,
        beta,
        f
    )

    x,esp=malha_base(
        delta,
        mat
    )

    risco=bool(
        item.get(
            "risco",
            alpha<1 or beta<1
        )
    )

    if not risco:
        return x,esp,mat

    candidatos=[
        dict(p)
        for p in (
            item.get(
                "candidatos",
                []
            ) or []
        )
    ]

    auditoria=item.get(
        "auditoria",
        {}
    ) or {}

    zeros_por_l={}

    for l in range(
        LMAX+1
    ):
        status=auditoria.get(
            l,
            auditoria.get(
                str(l),
                None
            )
        )

        if status=="ok":
            zeros_por_l[l]=[
                complex(
                    p["pole_real"],
                    p["pole_imag"]
                )
                for p in candidatos
                if int(
                    p.get(
                        "l",
                        -1
                    )
                )==l
                and bool(
                    p.get(
                        "argument_audit_ok",
                        False
                    )
                )
            ]
        else:
            zeros_por_l[l]=None

    modos=modos_por_l(
        delta,
        mat,
        zeros_por_l,
        x,
        esp
    )

    bons=[
        p
        for p in candidatos
        if bool(
            p.get(
                "argument_audit_ok",
                False
            )
        )
    ]

    laurent=(
        separacao_laurent(
            bons,
            delta,
            mat
        )
        if len(bons)>=2
        else None
    )

    vis=[
        p
        for p in bons
        if np.isfinite(
            p.get(
                "Q_pole",
                np.nan
            )
        )
        and float(
            p.get(
                "Q_pole",
                np.nan
            )
        )>=Q_MIN_POLO
        and np.isfinite(
            p.get(
                "prom_log10_pole",
                np.nan
            )
        )
        and float(
            p.get(
                "prom_log10_pole",
                np.nan
            )
        )>=PROM_WT
        and np.isfinite(
            p.get(
                "dx_nec",
                np.nan
            )
        )
        and float(
            p.get(
                "dx_nec",
                np.nan
            )
        )>0
    ]

    alvos=[
        (
            float(p["pole_real"]),
            float(p["dx_nec"]),
            None
        )
        for p in vis
    ]

    for m in modos:
        if (
            m.get(
                "N_zeros"
            ) is None
            or m[
                "regra"
            ] in (
                "proeminencia",
                "sem_polos"
            )
        ):
            P=np.sort(
                np.asarray(
                    m["x"],
                    float
                )
            )

            if len(P):
                viz=np.minimum(
                    np.r_[
                        np.inf,
                        np.diff(P)
                    ],
                    np.r_[
                        np.diff(P),
                        np.inf
                    ]
                )

                alvos += [
                    (
                        float(v),
                        0.5*min(
                            float(d),
                            0.25*esp
                        ),
                        None
                    )
                    for v,d in zip(
                        P,
                        viz
                    )
                ]

    xr,piso=malha_dx(
        x,
        alvos,
        esp,
        laurent
    )

    return xr,esp,mat

def picos_da_propria_malha(item,xr,esp,mat):
    delta=float(
        item["delta"]
    )

    WT=lambda v:energia_total(
        v,
        delta,
        mat
    )

    wr=WT(
        xr
    )

    idx_visiveis,_=maximos(
        WT,
        xr,
        esp,
        prom=PROM_WT,
        w=wr
    )

    return wr,idx_visiveis

def casar_ressonancias_salvas_com_maximos(
    item,
    xr,
    idx_visiveis
):
    salvos=item.get(
        "ressonancias_malha",
        []
    ) or []

    if len(
        salvos
    )==0 or len(
        idx_visiveis
    )==0:
        return []

    maximos_x=xr[
        idx_visiveis
    ]

    usados=set()
    marcadores=[]

    for ir,r in enumerate(
        salvos
    ):
        x0=float(
            r.get(
                "xM_peak",
                np.nan
            )
        )

        if not np.isfinite(
            x0
        ):
            continue

        j0=int(
            np.searchsorted(
                xr,
                x0
            )
        )

        j0=np.clip(
            j0,
            1,
            len(xr)-2
        )

        dx_esq=xr[j0]-xr[j0-1]
        dx_dir=xr[j0+1]-xr[j0]
        dx_local=max(
            dx_esq,
            dx_dir
        )

        dist=np.abs(
            maximos_x-x0
        )

        ordem=np.argsort(
            dist
        )

        escolhido=None

        for k in ordem:
            j=int(
                idx_visiveis[k]
            )

            if j in usados:
                continue

            if dist[k]<=3.0*dx_local:
                escolhido=j
                break

        if escolhido is not None:
            usados.add(
                escolhido
            )

            marcadores.append({
                "idx":escolhido,
                "x_plot":float(
                    xr[escolhido]
                ),
                "resonancia":r,
                "origem_marcador":"padrao"
            })

    return sorted(
        marcadores,
        key=lambda p:p[
            "x_plot"
        ]
    )

def fallback_picos_salvos_no_zoom(
    item,
    xr,
    wr,
    x_zoom_max
):
    salvos=item.get(
        "ressonancias_malha",
        []
    ) or []

    if not salvos:
        return []

    xs_gerador=np.asarray(
        item.get(
            "xs_malha_refinada",
            []
        ) or [],
        dtype=float
    )

    usados=set()
    marcadores=[]

    for ir,r in enumerate(
        salvos
    ):
        x_salvo=float(
            r.get(
                "xM_peak",
                np.nan
            )
        )

        if not np.isfinite(
            x_salvo
        ):
            continue

        if not (
            X_MIN<=x_salvo<=x_zoom_max
        ):
            continue

        if (
            ir<len(xs_gerador)
            and np.isfinite(
                xs_gerador[ir]
            )
        ):
            x_ref=float(
                xs_gerador[ir]
            )
        elif len(
            xs_gerador
        ):
            finite=np.isfinite(
                xs_gerador
            )

            if np.any(
                finite
            ):
                candidatos_x=xs_gerador[
                    finite
                ]

                x_ref=float(
                    candidatos_x[
                        np.argmin(
                            np.abs(
                                candidatos_x-x_salvo
                            )
                        )
                    ]
                )
            else:
                x_ref=x_salvo
        else:
            x_ref=x_salvo

        j0=int(
            np.argmin(
                np.abs(
                    xr-x_ref
                )
            )
        )

        lo=max(
            1,
            j0-4
        )

        hi=min(
            len(xr)-1,
            j0+5
        )

        candidatos=[]

        for j in range(
            lo,
            hi
        ):
            if j in usados:
                continue

            if not np.isfinite(
                wr[j]
            ):
                continue

            if (
                wr[j]>=wr[j-1]
                and wr[j]>=wr[j+1]
            ):
                candidatos.append(
                    j
                )

        if candidatos:
            escolhido=min(
                candidatos,
                key=lambda j:abs(
                    xr[j]-x_ref
                )
            )
        else:
            escolhido=j0

            if (
                escolhido<=0
                or escolhido>=len(xr)-1
                or escolhido in usados
                or not np.isfinite(
                    wr[escolhido]
                )
            ):
                continue

        usados.add(
            escolhido
        )

        marcadores.append({
            "idx":int(
                escolhido
            ),
            "x_plot":float(
                xr[escolhido]
            ),
            "resonancia":r,
            "origem_marcador":"fallback_xs_malha_refinada"
        })

    return sorted(
        marcadores,
        key=lambda p:p[
            "x_plot"
        ]
    )

def load_curves(
    checkpoint
):
    if not os.path.exists(
        checkpoint
    ):
        raise FileNotFoundError(
            checkpoint
        )

    with np.load(
        checkpoint,
        allow_pickle=True
    ) as data:
        curves=data[
            "curves"
        ].item()

    rows=[]

    for chave,item in curves.items():
        if item.get(
            "status"
        )!="ok":
            continue

        alpha=float(
            item.get(
                "alpha",
                np.nan
            )
        )

        beta=float(
            item.get(
                "beta",
                np.nan
            )
        )

        delta=float(
            item.get(
                "delta",
                np.nan
            )
        )

        if not (
            np.isfinite(alpha)
            and alpha>0
            and np.isfinite(beta)
            and beta>0
            and np.isfinite(delta)
        ):
            continue

        rows.append({
            "curve_key":chave,
            "alpha":alpha,
            "beta":beta,
            "eta":eta_from_alpha_beta(
                alpha,
                beta
            ),
            "delta":delta,
            "item":item
        })

    return pd.DataFrame(
        rows
    )

MANUAL_REPRESENTATIVE_PAIRS={
    "eta_positive":{
        "eta_label":r"$\eta_c\approx+0.7$",
        "alpha":0.006333333333333333,
        "beta":0.001
    },
    "eta_zero":{
        "eta_label":r"$\eta_c\approx0$",
        "alpha":0.008111111111111111,
        "beta":0.008111111111111111
    },
    "eta_negative":{
        "eta_label":r"$\eta_c\approx-0.7$",
        "alpha":0.001,
        "beta":0.006333333333333333
    }
}

TABLE_DELTAS=[
    0.10,
    0.50,
    1.35
]

def find_nearest_alpha_beta(
    curves_df,
    target_alpha,
    target_beta
):
    pairs=curves_df[
        [
            "alpha",
            "beta"
        ]
    ].drop_duplicates().reset_index(
        drop=True
    )

    target=np.log(
        np.array([
            target_alpha,
            target_beta
        ])
    )

    vals=np.log(
        pairs.to_numpy(
            float
        )
    )

    i=int(
        np.argmin(
            np.sum(
                (vals-target)**2,
                axis=1
            )
        )
    )

    return (
        float(
            pairs.loc[
                i,
                "alpha"
            ]
        ),
        float(
            pairs.loc[
                i,
                "beta"
            ]
        )
    )

def select_nearest_delta_rows(
    curves_df,
    alpha,
    beta,
    target_deltas
):
    pair_df=curves_df.loc[
        np.isclose(
            curves_df[
                "alpha"
            ],
            alpha,
            rtol=1e-9
        )
        &np.isclose(
            curves_df[
                "beta"
            ],
            beta,
            rtol=1e-9
        )
    ]

    out=[]

    for d in target_deltas:
        if len(
            pair_df
        )==0:
            out.append(
                None
            )
            continue

        i=int(
            np.argmin(
                np.abs(
                    pair_df[
                        "delta"
                    ].to_numpy(
                        float
                    )-d
                )
            )
        )

        out.append(
            pair_df.iloc[
                i
            ]
        )

    return out

FIGSIZE_CURVES=(
    6.5,
    5.0
)

COLORS=[
    "#0f223d",
    "#810000",
    "black"
]

LINESTYLES=[
    "-",
    "--",
    "-."
]

ZOOM_POR_REGIME={
    "velocity":(
        0.010,
        [
            0.0000,
            0.0025,
            0.0050,
            0.0075,
            0.0100
        ]
    ),
    "mixed":(
        0.020,
        [
            0.000,
            0.005,
            0.010,
            0.015,
            0.020
        ]
    ),
    "density":(
        1.000,
        [
            0.0,
            0.2,
            0.4,
            0.6,
            0.8,
            1.0
        ]
    )
}

Y_TICKS=np.array([
    1e-7,
    1e-3,
    1e1,
    1e5,
    1e9
])

YLIM=(
    1e-7,
    1e10
)

def preparar_curva(
    row,
    f,
    x_zoom_max
):
    if row is None:
        return None

    item=row[
        "item"
    ]

    xr,esp,mat=reconstruir_malha(
        item,
        f
    )

    wr,idx_visiveis=picos_da_propria_malha(
        item,
        xr,
        esp,
        mat
    )

    marcadores_padrao=casar_ressonancias_salvas_com_maximos(
        item,
        xr,
        idx_visiveis
    )

    picos_zoom=[
        p
        for p in marcadores_padrao
        if X_MIN<=p[
            "x_plot"
        ]<=x_zoom_max
    ]

    # ============================================================
    # ÚNICA CORREÇÃO:
    # se a lógica anterior não marcou NENHUM pico no zoom,
    # usa as posições já salvas pelo gerador em
    # xs_malha_refinada e encaixa na própria malha reconstruída.
    #
    # Se a lógica anterior funcionou, NÃO MUDA NADA.
    # ============================================================

    usou_fallback=False

    if len(
        picos_zoom
    )==0:
        salvos_zoom=[
            p
            for p in (
                item.get(
                    "ressonancias_malha",
                    []
                ) or []
            )
            if np.isfinite(
                p.get(
                    "xM_peak",
                    np.nan
                )
            )
            and X_MIN<=float(
                p.get(
                    "xM_peak",
                    np.nan
                )
            )<=x_zoom_max
        ]

        if len(
            salvos_zoom
        ):
            fallback=fallback_picos_salvos_no_zoom(
                item,
                xr,
                wr,
                x_zoom_max
            )

            if len(
                fallback
            ):
                picos_zoom=fallback
                usou_fallback=True

    picos_zoom=sorted(
        picos_zoom,
        key=lambda p:p[
            "x_plot"
        ]
    )

    return {
        "row":row,
        "item":item,
        "xr":xr,
        "wr":wr,
        "esp":esp,
        "mat":mat,
        "idx_visiveis":idx_visiveis,
        "marcadores_padrao":marcadores_padrao,
        "picos_zoom":picos_zoom,
        "usou_fallback":usou_fallback
    }

def ressonancias_malha_todas(
    item
):
    picos=item.get(
        "ressonancias_malha",
        []
    ) or []

    picos=[
        p
        for p in picos
        if np.isfinite(
            p.get(
                "xM_peak",
                np.nan
            )
        )
        and X_MIN<=float(
            p.get(
                "xM_peak",
                np.nan
            )
        )<=X_MAX
    ]

    return sorted(
        picos,
        key=lambda p:float(
            p.get(
                "xM_peak",
                np.inf
            )
        )
    )

def resumo_picos_malha(
    preparado
):
    if preparado is None:
        return None

    picos_zoom=preparado[
        "picos_zoom"
    ]

    if len(
        picos_zoom
    ):
        primeiro=picos_zoom[
            0
        ]

        res_primeiro=primeiro[
            "resonancia"
        ]

        # POSIÇÃO = bolinha realmente mostrada
        first_x=float(
            primeiro[
                "x_plot"
            ]
        )

        # FWHM/Q = mesma ressonância salva no gerador
        first_fwhm=float(
            res_primeiro.get(
                "FWHM_energy",
                np.nan
            )
        )

        first_q=float(
            res_primeiro.get(
                "Q_energy",
                np.nan
            )
        )

    else:
        first_x=np.nan
        first_fwhm=np.nan
        first_q=np.nan

    # ============================================================
    # MACRO-FSR E MÉDIAS:
    # TODAS AS RESSONANCIAS_MALHA SALVAS ATÉ x_M=1.
    # ============================================================

    todos=ressonancias_malha_todas(
        preparado[
            "item"
        ]
    )

    if len(
        todos
    ):
        x=np.asarray(
            [
                p.get(
                    "xM_peak",
                    np.nan
                )
                for p in todos
            ],
            dtype=float
        )

        fwhm=np.asarray(
            [
                p.get(
                    "FWHM_energy",
                    np.nan
                )
                for p in todos
            ],
            dtype=float
        )

        q=np.asarray(
            [
                p.get(
                    "Q_energy",
                    np.nan
                )
                for p in todos
            ],
            dtype=float
        )

    else:
        x=np.array(
            [],
            dtype=float
        )

        fwhm=np.array(
            [],
            dtype=float
        )

        q=np.array(
            [],
            dtype=float
        )

    vf=fwhm[
        np.isfinite(
            fwhm
        )
    ]

    vq=q[
        np.isfinite(
            q
        )
    ]

    gaps=(
        np.diff(
            x
        )
        if len(x)>=2
        else np.array(
            [],
            dtype=float
        )
    )

    return {
        "Actual_Delta":float(
            preparado[
                "row"
            ][
                "delta"
            ]
        ),
        "N_peaks":len(
            todos
        ),
        "First_Peak_xM":first_x,
        "First_FWHM":first_fwhm,
        "First_Q":first_q,
        "Macro_FSR_Median":(
            float(
                np.median(
                    gaps
                )
            )
            if len(gaps)
            else np.nan
        ),
        "Mean_FWHM":(
            float(
                np.mean(
                    vf
                )
            )
            if len(vf)
            else np.nan
        ),
        "Std_FWHM":(
            float(
                np.std(
                    vf,
                    ddof=1
                )
            )
            if len(vf)>1
            else (
                0.0
                if len(vf)==1
                else np.nan
            )
        ),
        "Mean_Q":(
            float(
                np.mean(
                    vq
                )
            )
            if len(vq)
            else np.nan
        ),
        "Std_Q":(
            float(
                np.std(
                    vq,
                    ddof=1
                )
            )
            if len(vq)>1
            else (
                0.0
                if len(vq)==1
                else np.nan
            )
        )
    }

TABLE_FIGSIZE_PEAKS=(
    6.4,
    4.5
)

TABLE_SPACER_WIDTH=0.013

row_labels=[
    "1st Peak\n"+r"($x_M$)",
    "1st Peak\nFWHM",
    "1st Peak\n"+r"$Q$-factor",
    "Macro-FSR\n"+r"($\Delta x_M$)",
    "Avg. FWHM\n"+r"($\mu\pm\sigma$) [$\times10^{-4}$]",
    "Avg. $Q$-factor\n"+r"($\mu\pm\sigma$)"
]

def latex_sci(
    value,
    sig=3
):
    if not np.isfinite(
        value
    ):
        return "--"

    if np.isclose(
        value,
        0
    ):
        return "0"

    e=int(
        np.floor(
            np.log10(
                abs(value)
            )
        )
    )

    m=value/(
        10**e
    )

    return rf"${m:.{sig-1}f}\times10^{{{e}}}$"

def fmt_peak_x(v):
    return (
        "--"
        if not np.isfinite(
            v
        )
        else f"{v:.5g}"
    )

def fmt_fwhm(v):
    return latex_sci(
        v,
        3
    )

def fmt_q(v):
    if not np.isfinite(
        v
    ):
        return "--"

    if (
        abs(v)>=1e3
        or (
            0<abs(v)<1e-2
        )
    ):
        return latex_sci(
            v,
            3
        )

    return f"{v:.3g}"

def fmt_macro(v):
    return latex_sci(
        v,
        3
    )

def fmt_mean_std(
    m,
    s,
    d=2
):
    if not np.isfinite(
        m
    ):
        return "--"

    if not np.isfinite(
        s
    ):
        s=0

    return (
        f"${m:.{d}f}$\n"
        +rf"$\pm{s:.{d}f}$"
    )

def fmt_mean_std_sci(
    m,
    s
):
    if not np.isfinite(
        m
    ):
        return "--"

    if not np.isfinite(
        s
    ):
        s=0

    linha=latex_sci(
        m,
        2
    )

    if np.isclose(
        s,
        0
    ):
        desvio=r"$\pm0$"
    else:
        e=int(
            np.floor(
                np.log10(
                    abs(s)
                )
            )
        )

        mant=s/(
            10**e
        )

        desvio=rf"$\pm{mant:.1f}\times10^{{{e}}}$"

    return (
        linha
        +"\n"
        +desvio
    )

def make_table_data(
    summaries
):
    return [
        [
            fmt_peak_x(
                s["First_Peak_xM"]
            ) if s else "--"
            for s in summaries
        ],
        [
            fmt_fwhm(
                s["First_FWHM"]
            ) if s else "--"
            for s in summaries
        ],
        [
            fmt_q(
                s["First_Q"]
            ) if s else "--"
            for s in summaries
        ],
        [
            fmt_macro(
                s["Macro_FSR_Median"]
            ) if s else "--"
            for s in summaries
        ],
        [
            fmt_mean_std(
                s["Mean_FWHM"]*1e4,
                s["Std_FWHM"]*1e4
            ) if s else "--"
            for s in summaries
        ],
        [
            fmt_mean_std_sci(
                s["Mean_Q"],
                s["Std_Q"]
            ) if s else "--"
            for s in summaries
        ]
    ]

def save_table(
    filename,
    data
):
    with plt.rc_context({
        "font.family":"STIXGeneral",
        "mathtext.fontset":"stix",
        "font.size":30,
        "figure.facecolor":"white",
        "axes.facecolor":"white",
        "savefig.facecolor":"white",
        "savefig.edgecolor":"white"
    }):
        fig,ax=plt.subplots(
            figsize=TABLE_FIGSIZE_PEAKS,
            facecolor="white"
        )

        ax.axis(
            "off"
        )

        table_data=[
            [
                row_labels[i],
                data[i][0],
                "",
                data[i][1],
                "",
                data[i][2]
            ]
            for i in range(
                len(row_labels)
            )
        ]

        table=ax.table(
            cellText=table_data,
            colLabels=[
                "Parameter",
                r"$\delta=0.10$",
                "",
                r"$\delta=0.50$",
                "",
                r"$\delta=1.35$"
            ],
            cellLoc="center",
            colWidths=[
                0.3600,
                0.2047,
                TABLE_SPACER_WIDTH,
                0.2047,
                TABLE_SPACER_WIDTH,
                0.2046
            ],
            bbox=[
                0.02,
                0.02,
                0.96,
                0.96
            ]
        )

        table.auto_set_font_size(
            False
        )

        table.set_fontsize(
            20
        )

        spacer_cols=(
            2,
            4
        )

        value_cols=(
            1,
            3,
            5
        )

        for (row,col),cell in table.get_celld().items():
            cell.visible_edges=""
            cell.set_edgecolor(
                "black"
            )
            cell.set_linewidth(
                0.0
            )
            cell.set_facecolor(
                "white"
            )
            cell.PAD=0.01

            text=cell.get_text()
            text.set_color(
                "black"
            )
            text.set_va(
                "center"
            )
            text.set_linespacing(
                0.90
            )

            if col==0:
                text.set_ha(
                    "left"
                )
                text.set_fontsize(
                    19
                )

            elif col in spacer_cols:
                text.set_text(
                    ""
                )

            else:
                text.set_ha(
                    "center"
                )
                text.set_fontsize(
                    20
                )

        for col in range(6):
            table[
                (0,col)
            ].visible_edges="TB"

            table[
                (0,col)
            ].set_linewidth(
                1.5
            )

        table[
            (0,0)
        ].get_text().set_fontweight(
            "bold"
        )

        for col in range(6):
            table[
                (3,col)
            ].visible_edges="B"

            table[
                (3,col)
            ].set_linewidth(
                1.1
            )

            table[
                (6,col)
            ].visible_edges="B"

            table[
                (6,col)
            ].set_linewidth(
                1.5
            )

        for row in range(
            1,
            7
        ):
            table[
                (row,0)
            ].get_text().set_fontsize(
                19
            )

        for row in (
            1,
            3
        ):
            for col in value_cols:
                table[
                    (row,col)
                ].get_text().set_fontsize(
                    20
                )

        for col in value_cols:
            table[
                (2,col)
            ].get_text().set_fontsize(
                18.2
            )

            table[
                (4,col)
            ].get_text().set_fontsize(
                18.2
            )

            table[
                (5,col)
            ].get_text().set_fontsize(
                19
            )

            table[
                (6,col)
            ].get_text().set_fontsize(
                18.2
            )

        fig.subplots_adjust(
            left=0.005,
            right=0.995,
            bottom=0.005,
            top=0.995
        )

        fig.savefig(
            filename+".png",
            dpi=600,
            bbox_inches="tight",
            pad_inches=0.01,
            facecolor="white",
            edgecolor="white"
        )

        fig.savefig(
            filename+".pdf",
            bbox_inches="tight",
            pad_inches=0.01,
            facecolor="white",
            edgecolor="white"
        )

        fig.savefig(
            filename+".svg",
            bbox_inches="tight",
            pad_inches=0.01,
            facecolor="white",
            edgecolor="white"
        )

        plt.show()
        plt.close(fig)

def plot_case(
    regime,
    case_name,
    eta_label,
    preparados,
    filename
):
    X_ZOOM_MAX,x_ticks=ZOOM_POR_REGIME[
        regime
    ]

    fig,ax=plt.subplots(
        figsize=FIGSIZE_CURVES
    )

    for i,preparado in enumerate(
        preparados
    ):
        if preparado is None:
            continue

        row=preparado[
            "row"
        ]

        xr=preparado[
            "xr"
        ]

        wr=preparado[
            "wr"
        ]

        picos_zoom=preparado[
            "picos_zoom"
        ]

        # ============================================================
        # LINHA:
        # exatamente a malha reconstruída.
        # NÃO adiciona xM_peak dentro dela.
        # ============================================================

        mask=(
            (xr>=X_MIN)
            &(xr<=X_ZOOM_MAX)
        )

        ax.plot(
            xr[
                mask
            ],
            wr[
                mask
            ],
            color=COLORS[
                i%len(COLORS)
            ],
            linestyle=LINESTYLES[
                i%len(LINESTYLES)
            ],
            linewidth=2.0,
            label=rf"$\delta={row['delta']:.2f}$"
        )

        # ============================================================
        # BOLINHAS:
        # sempre em pontos da MESMA linha/malha plotada.
        # ============================================================

        idx_zoom=np.asarray(
            [
                p["idx"]
                for p in picos_zoom
            ],
            dtype=int
        )

        if len(
            idx_zoom
        ):
            ax.scatter(
                xr[
                    idx_zoom
                ],
                wr[
                    idx_zoom
                ],
                s=24,
                color=COLORS[
                    i%len(COLORS)
                ],
                edgecolors="white",
                linewidths=0.45,
                zorder=5
            )

        fallback_txt=(
            " SIM"
            if preparado[
                "usou_fallback"
            ]
            else " nao"
        )

        print(
            f"  delta={row['delta']:.2f} | "
            f"pontos malha={len(xr)} | "
            f"picos prom>=0.1={len(preparado['idx_visiveis'])} | "
            f"marcadores no zoom={len(idx_zoom)} | "
            f"fallback gerador={fallback_txt}"
        )

        if len(
            picos_zoom
        ):
            print(
                "    primeiro marcador = "
                f"{picos_zoom[0]['x_plot']:.10g}"
            )

    ax.set_yscale(
        "log"
    )

    ax.set_xlim(
        0.0,
        X_ZOOM_MAX
    )

    ax.set_ylim(
        *YLIM
    )

    ax.xaxis.set_major_locator(
        FixedLocator(
            np.asarray(
                x_ticks,
                float
            )
        )
    )

    ax.yaxis.set_major_locator(
        FixedLocator(
            Y_TICKS
        )
    )

    ax.minorticks_off()

    ax.yaxis.set_minor_formatter(
        NullFormatter()
    )

    ax.set_xlabel(
        r"$x_M=k_Ma$",
        fontsize=20,
        labelpad=8
    )

    ax.tick_params(
        direction="in",
        length=5,
        width=1.2,
        labelsize=16,
        pad=6
    )

    for spine in ax.spines.values():
        spine.set_linewidth(
            1.8
        )
        spine.set_edgecolor(
            "black"
        )

    ax.grid(
        False
    )

    legend=ax.legend(
        loc="lower right",
        fontsize=20,
        frameon=True,
        handlelength=2.5,
        borderpad=0.50,
        labelspacing=0.40,
        handletextpad=0.65
    )

    legend.get_frame().set(
        edgecolor="black",
        facecolor="white",
        linewidth=1.8,
        alpha=1.0
    )

    ax.text(
        0.965,
        0.94,
        eta_label,
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=16,
        color="black",
        bbox={
            "boxstyle":"square,pad=0.4",
            "facecolor":"white",
            "edgecolor":"black",
            "linewidth":2.0,
            "alpha":1.0
        }
    )

    fig.subplots_adjust(
        left=0.20,
        right=0.97,
        bottom=0.18,
        top=0.96
    )

    fig.savefig(
        filename+".png",
        dpi=1200,
        bbox_inches="tight",
        pad_inches=0.05
    )

    fig.savefig(
        filename+".pdf",
        bbox_inches="tight",
        pad_inches=0.05
    )

    plt.show()
    plt.close(fig)

def main():
    global NORMALIZACAO

    for regime,case in CASES.items():
        NORMALIZACAO=ler_normalizacao(
            case[
                "param"
            ]
        )

        X_ZOOM_MAX,_=ZOOM_POR_REGIME[
            regime
        ]

        print(
            "\n"
            +"="*100
        )

        print(
            f"{regime.upper()} | "
            f"f={case['f']:.2f} | "
            f"normalizacao={NORMALIZACAO} | "
            f"x grafico=0 -> {X_ZOOM_MAX}"
        )

        print(
            "="*100
        )

        curves_df=load_curves(
            case[
                "checkpoint"
            ]
        )

        output_dir=os.path.join(
            OUT_DIR,
            regime
        )

        os.makedirs(
            output_dir,
            exist_ok=True
        )

        for case_name,info in MANUAL_REPRESENTATIVE_PAIRS.items():
            alpha,beta=find_nearest_alpha_beta(
                curves_df,
                info[
                    "alpha"
                ],
                info[
                    "beta"
                ]
            )

            rows=select_nearest_delta_rows(
                curves_df,
                alpha,
                beta,
                TABLE_DELTAS
            )

            print(
                f"\n{case_name} | "
                f"alpha={alpha:.6g} | "
                f"beta={beta:.6g} | "
                f"eta={eta_from_alpha_beta(alpha,beta):+.3f}"
            )

            preparados=[
                preparar_curva(
                    row,
                    case[
                        "f"
                    ],
                    X_ZOOM_MAX
                )
                for row in rows
            ]

            summaries=[
                resumo_picos_malha(
                    preparado
                )
                for preparado in preparados
            ]

            print(
                "\nTabela:"
            )

            for d,s,preparado in zip(
                TABLE_DELTAS,
                summaries,
                preparados
            ):
                if s is None:
                    print(
                        f"  delta={d:.2f}: sem dados"
                    )
                    continue

                n_figura=(
                    len(
                        preparado[
                            "picos_zoom"
                        ]
                    )
                    if preparado is not None
                    else 0
                )

                print(
                    f"  delta={d:.2f} | "
                    f"1o pico={s['First_Peak_xM']:.10g} | "
                    f"N figura={n_figura} | "
                    f"N usado nas medias ate xM=1={s['N_peaks']}"
                )

            table_data=make_table_data(
                summaries
            )

            save_table(
                os.path.join(
                    output_dir,
                    f"table_{case_name}_compact"
                ),
                table_data
            )

            plot_case(
                regime,
                case_name,
                info[
                    "eta_label"
                ],
                preparados,
                os.path.join(
                    output_dir,
                    f"WT_internal_vs_xM_{case_name}"
                )
            )

        print(
            "\nSaída:",
            output_dir
        )

if __name__=="__main__":
    main()

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap,FuncNorm
from matplotlib.patches import Rectangle
from mpl_toolkits.axes_grid1 import make_axes_locatable
BASE_DIR=DADOS

CASES={
    "velocity":{
        "f":0.00,
        "checkpoint":os.path.join(
            BASE_DIR,
            "coreshell_lmax10_mixed_mix0.00",
            "checkpoint_coreshell_lmax10_mixed_mix0.00.npz"
        ),
        "colorbar_max":None
    },
    "mixed":{
        "f":0.50,
        "checkpoint":os.path.join(
            BASE_DIR,
            "coreshell_lmax10_mixed_mix0.50",
            "checkpoint_coreshell_lmax10_mixed_mix0.50.npz"
        ),
        "colorbar_max":None
    },
    "density":{
        "f":1.00,
        "checkpoint":os.path.join(
            BASE_DIR,
            "coreshell_lmax10_mixed_mix1.00",
            "checkpoint_coreshell_lmax10_mixed_mix1.00.npz"
        ),
        "colorbar_max":3.0
    }
}

BOX_COLOR="#600B00"
BOX_LINEWIDTH=1.8
CBAR_BOX_LINEWIDTH=1.8
CELL_LINECOLOR="white"
CELL_LINEWIDTH=0.42
CENTER_LINE_COLOR="#C7D0DB"
CENTER_LINE_STYLE="--"
CENTER_LINE_WIDTH=1.35
FIG_FACE_COLOR="white"
FIGSIZE_MAP=(11,13)
LABEL_FONTSIZE=30
TICK_FONTSIZE=24

plt.rcParams.update({
    "font.size":TICK_FONTSIZE,
    "axes.labelsize":LABEL_FONTSIZE,
    "xtick.labelsize":TICK_FONTSIZE,
    "ytick.labelsize":TICK_FONTSIZE
})

MEAN_RESONANCE_CMAP=LinearSegmentedColormap.from_list(
    "mean_resonances",
    [
        (0.0,"white"),
        (0.5,"#526F92"),
        (1.0,"black")
    ],
    N=256
)

MEAN_RESONANCE_CMAP.set_bad("#D9D9D9")

def nearest_index(value,array):
    return int(
        np.argmin(
            np.abs(
                np.asarray(array,dtype=float)-float(value)
            )
        )
    )

def eta_bin_indices_generator(eta_values,eta_bins):
    eta_values=np.asarray(eta_values,dtype=float)
    eta_clipped=np.clip(
        eta_values,
        eta_bins[0],
        eta_bins[-1]
    )

    indices=np.searchsorted(
        eta_bins,
        eta_clipped,
        side="left"
    )-1

    return np.clip(
        indices,
        0,
        len(eta_bins)-2
    )

def eta_bin_index(eta_value,eta_bins):
    return int(
        eta_bin_indices_generator(
            np.array([eta_value]),
            eta_bins
        )[0]
    )

def style_axes_and_cbar(ax,cbar=None):
    ax.tick_params(
        axis="both",
        labelsize=TICK_FONTSIZE,
        width=1.25,
        length=4.5,
        direction="out"
    )

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor(BOX_COLOR)
        spine.set_linewidth(BOX_LINEWIDTH)

    if cbar is not None:
        cbar.ax.tick_params(
            labelsize=TICK_FONTSIZE,
            width=1.25,
            length=4.5,
            direction="out"
        )

        for spine in cbar.ax.spines.values():
            spine.set_visible(True)
            spine.set_edgecolor(BOX_COLOR)
            spine.set_linewidth(CBAR_BOX_LINEWIDTH)

def load_checkpoint_vetorizado(filepath):
    if not os.path.exists(filepath):
        raise FileNotFoundError(
            f"File not found:\n{filepath}"
        )

    with np.load(filepath,allow_pickle=True) as data:
        if "curves" not in data.files:
            raise KeyError(
                "Este arquivo não parece ser o checkpoint novo do core-shell: "
                f"a chave 'curves' não foi encontrada.\n"
                f"Chaves encontradas: {data.files}"
            )

        curves=data["curves"].item()

        if "delta_values" in data.files:
            delta_vals_saved=np.asarray(
                data["delta_values"],
                dtype=float
            )
        elif "delta_vals" in data.files:
            delta_vals_saved=np.asarray(
                data["delta_vals"],
                dtype=float
            )
        else:
            delta_vals_saved=None

    alphas=[]
    betas=[]
    deltas=[]
    n_peaks=[]
    statuses=[]

    for item in curves.values():
        alpha=float(item.get("alpha",np.nan))
        beta=float(item.get("beta",np.nan))
        delta=float(item.get("delta",np.nan))
        status=str(item.get("status",""))

        if not (
            np.isfinite(alpha)
            and alpha>0.0
            and np.isfinite(beta)
            and beta>0.0
            and np.isfinite(delta)
        ):
            continue

        if status=="ok":

            # SOMENTE CONTAGEM PELA MALHA DE W_T
            # normal -> malha base
            # risco -> malha refinada
            ressonancias_malha=item.get(
                "ressonancias_malha",
                []
            ) or []

            n_res=float(
                len(
                    ressonancias_malha
                )
            )

        else:
            n_res=np.nan

        alphas.append(alpha)
        betas.append(beta)
        deltas.append(delta)
        n_peaks.append(n_res)
        statuses.append(status)

    alphas=np.asarray(alphas,dtype=float)
    betas=np.asarray(betas,dtype=float)
    deltas=np.asarray(deltas,dtype=float)
    n_peaks=np.asarray(n_peaks,dtype=float)
    statuses=np.asarray(statuses,dtype=object)

    if delta_vals_saved is not None:
        delta_vals=delta_vals_saved
    else:
        delta_vals=np.unique(deltas)

    return {
        "delta_vals":delta_vals,
        "alpha":alphas,
        "beta":betas,
        "delta":deltas,
        "n_peaks":n_peaks,
        "status":statuses
    }

def project_peaks_to_delta_eta(
    db,
    eta_bins=np.linspace(-1.0,1.0,21)
):
    eta_bins=np.asarray(
        eta_bins,
        dtype=float
    )

    eta_centers=0.5*(
        eta_bins[:-1]+eta_bins[1:]
    )

    alpha=db["alpha"]
    beta=db["beta"]
    delta=db["delta"]
    n_peaks=db["n_peaks"]

    eta_points=(
        alpha-beta
    )/(
        alpha+beta
    )

    delta_vals=np.asarray(
        db["delta_vals"],
        dtype=float
    )

    n_delta=len(delta_vals)
    n_eta=len(eta_centers)

    mean_map=np.full(
        (n_delta,n_eta),
        np.nan,
        dtype=float
    )

    samples_map=np.zeros(
        (n_delta,n_eta),
        dtype=int
    )

    point_eta_index=eta_bin_indices_generator(
        eta_points,
        eta_bins
    )

    point_delta_index=np.array(
        [
            nearest_index(
                d,
                delta_vals
            )
            for d in delta
        ],
        dtype=int
    )

    valid=(
        np.isfinite(n_peaks)
        &np.isfinite(eta_points)
    )

    for idd in range(n_delta):
        mask_delta=(
            valid
            &(point_delta_index==idd)
        )

        for i_eta in range(n_eta):
            mask=(
                mask_delta
                &(point_eta_index==i_eta)
            )

            if not np.any(mask):
                continue

            values=n_peaks[mask]

            samples_map[
                idd,
                i_eta
            ]=len(values)

            # O MAPA É DA MÉDIA DO NÚMERO DE RESSONÂNCIAS
            mean_map[
                idd,
                i_eta
            ]=np.mean(values)

    return (
        eta_bins,
        eta_centers,
        mean_map,
        samples_map
    )

def log_color_norm(vmax):
    vmax=float(vmax)

    if not np.isfinite(vmax) or vmax<=0:
        vmax=1.0

    def forward(x):
        x=np.asarray(x,dtype=float)
        return np.log1p(
            np.clip(
                x,
                0.0,
                None
            )
        )

    def inverse(y):
        y=np.asarray(y,dtype=float)
        return np.expm1(y)

    return FuncNorm(
        (forward,inverse),
        vmin=0.0,
        vmax=vmax,
        clip=True
    )

def nice_125_candidates(vmax):
    if vmax<=0:
        return np.array([],dtype=float)

    emin=-6
    emax=int(
        np.ceil(
            np.log10(vmax)
        )
    )+1

    values=[]

    for exponent in range(emin,emax+1):
        base=10.0**exponent

        for factor in (1.0,2.0,5.0):
            value=factor*base

            if 0<value<vmax:
                values.append(value)

    return np.asarray(
        sorted(
            set(values)
        ),
        dtype=float
    )

def nearest_nice_value(target,candidates):
    if len(candidates)==0:
        return target

    target=max(float(target),np.finfo(float).tiny)

    distance=np.abs(
        np.log(candidates)
        -np.log(target)
    )

    return float(
        candidates[
            np.argmin(distance)
        ]
    )

def clean_colorbar_ticks(vmax,max_ticks=5):
    vmax=float(vmax)

    if not np.isfinite(vmax) or vmax<=0:
        return np.array(
            [0.0],
            dtype=float
        )

    # DENSITY 0 -> 3:
    # fica exatamente 0, 1, 2, 3
    if np.isclose(vmax,3.0):
        return np.array(
            [0.0,1.0,2.0,3.0],
            dtype=float
        )

    # Casos pequenos
    if vmax<=4.0 and np.isclose(vmax,round(vmax)):
        return np.arange(
            0,
            int(round(vmax))+1,
            dtype=float
        )

    candidates=nice_125_candidates(vmax)

    # As posições desejadas são escolhidas NA ESCALA DE COR log(1+N),
    # para ficarem visualmente bem separadas.
    fractions=np.array(
        [0.25,0.50,0.75],
        dtype=float
    )

    transformed_max=np.log1p(vmax)

    targets=np.expm1(
        fractions*transformed_max
    )

    interior=[]

    for target in targets:
        value=nearest_nice_value(
            target,
            candidates
        )

        # não deixa o primeiro número encostado no zero
        relative_position=(
            np.log1p(value)
            /transformed_max
        )

        if relative_position<0.18:
            continue

        # não coloca um número praticamente colado no máximo
        if np.log1p(value)/transformed_max>0.88:
            continue

        if value>0 and value<vmax:
            interior.append(value)

    interior=np.unique(
        np.asarray(
            interior,
            dtype=float
        )
    )

    ticks=np.r_[
        0.0,
        interior,
        vmax
    ]

    ticks=np.unique(
        ticks
    )

    if len(ticks)>max_ticks:
        ticks=np.r_[
            ticks[0],
            ticks[
                np.linspace(
                    1,
                    len(ticks)-2,
                    max_ticks-2
                ).round().astype(int)
            ],
            ticks[-1]
        ]

    return np.asarray(
        ticks,
        dtype=float
    )

def format_colorbar_tick(value,vmax):
    if np.isclose(value,0.0):
        return "0"

    if np.isclose(value,round(value)):
        return f"{int(round(value))}"

    if vmax<=10:
        return f"{value:.2g}"

    if vmax<=100:
        return f"{value:.1f}"

    return f"{value:.0f}"

def get_colorbar_max(metric_map,forced_max=None):
    finite=np.asarray(
        metric_map,
        dtype=float
    )

    finite=finite[
        np.isfinite(finite)
    ]

    if finite.size==0:
        raise RuntimeError(
            "Nenhum valor finito disponível no mapa."
        )

    map_max=float(
        np.max(finite)
    )

    if forced_max is not None:
        vmax=float(forced_max)

        if map_max>vmax+1e-12:
            raise RuntimeError(
                f"O mapa possui média máxima de {map_max:g}, "
                f"mas a colorbar foi limitada a {vmax:g}."
            )
    else:
        # IMPORTANTE:
        # o mapa mostra MÉDIA, então o máximo da colorbar
        # vem da própria média e NÃO do maior N_res de uma curva.
        vmax=map_max

    if vmax<=0:
        vmax=1.0

    return vmax,map_max

def eta_do_par_mais_proximo(db,alpha_alvo,beta_alvo):
    validos=np.isfinite(db["n_peaks"])
    pares=np.unique(np.column_stack([db["alpha"][validos],db["beta"][validos]]),axis=0)
    alvo=np.log(np.array([alpha_alvo,beta_alvo],dtype=float))
    i=int(np.argmin(np.sum((np.log(pares)-alvo)**2,axis=1)))
    alpha,beta=pares[i]
    return float((alpha-beta)/(alpha+beta))


def plot_delta_eta_metric(
    db,
    eta_bins,
    eta_centers,
    metric_map,
    metric_name,
    cbar_label,
    output_name,
    out_dir,
    forced_colorbar_max=None
):
    delta_vals=np.asarray(
        db["delta_vals"],
        dtype=float
    )

    nd=len(delta_vals)
    n_eta=len(eta_centers)

    valid_cells=np.isfinite(
        metric_map
    )

    if not np.any(valid_cells):
        raise RuntimeError(
            f"No completed data are available to build the map: {metric_name}"
        )

    vmax,map_max=get_colorbar_max(
        metric_map,
        forced_max=forced_colorbar_max
    )

    # SOMENTE A ESCALA DE COR É LOGARÍTMICA.
    # metric_map NÃO É TRANSFORMADO.
    norm=log_color_norm(vmax)

    cbar_ticks=clean_colorbar_ticks(
        vmax,
        max_ticks=5
    )

    x_edges=np.arange(
        nd+1,
        dtype=float
    )-0.5

    y_edges=np.arange(
        n_eta+1,
        dtype=float
    )-0.5

    fig=plt.figure(
        figsize=FIGSIZE_MAP
    )

    fig.patch.set_facecolor(
        FIG_FACE_COLOR
    )

    ax=fig.add_axes([
        0.105,
        0.115,
        0.725,
        0.725
    ])

    # DADOS ORIGINAIS DA MÉDIA.
    # O LOG ESTÁ APENAS EM norm.
    mesh=ax.pcolormesh(
        x_edges,
        y_edges,
        metric_map.T,
        cmap=MEAN_RESONANCE_CMAP,
        norm=norm,
        shading="flat",
        edgecolors=CELL_LINECOLOR,
        linewidths=CELL_LINEWIDTH
    )

    # eta real dos pares usados nas curvas (par do dataset mais proximo do alvo, como nas celulas das curvas)
    eta_red=eta_do_par_mais_proximo(db,0.006333333333333333,0.001)
    eta_black=eta_do_par_mais_proximo(db,0.008111111111111111,0.008111111111111111)
    eta_gray=eta_do_par_mais_proximo(db,0.001,0.006333333333333333)

    idx_eta_red=eta_bin_index(
        eta_red,
        eta_bins
    )

    idx_eta_black=eta_bin_index(
        eta_black,
        eta_bins
    )

    idx_eta_gray=eta_bin_index(
        eta_gray,
        eta_bins
    )

    deltas_alvo=[
        0.10,
        0.50,
        1.35
    ]

    idx_deltas=[
        nearest_index(
            delta,
            delta_vals
        )
        for delta in deltas_alvo
    ]

    highlights=[
        (
            idx_eta_red,
            BOX_COLOR
        ),
        (
            idx_eta_black,
            "#B0B0B0"
        ),
        (
            idx_eta_gray,
            "black"
        )
    ]

    for idx_delta in idx_deltas:
        for idx_eta,color in highlights:
            rect=Rectangle(
                (
                    idx_delta-0.5,
                    idx_eta-0.5
                ),
                1.0,
                1.0,
                fill=False,
                edgecolor=color,
                linewidth=5,
                zorder=10
            )

            ax.add_patch(rect)

    nice_deltas=[
        0.01,
        0.20,
        0.50,
        0.80,
        1.10,
        1.50
    ]

    tick_d_indices=[
        nearest_index(
            value,
            delta_vals
        )
        for value in nice_deltas
    ]

    ax.set_xticks(
        tick_d_indices
    )

    ax.set_xticklabels(
        [
            f"{delta_vals[index]:g}"
            for index in tick_d_indices
        ]
    )

    nice_etas=np.array(
        [
            -1.0,
            -0.5,
            0.0,
            0.5,
            1.0
        ],
        dtype=float
    )

    eta_step=(
        eta_bins[1]
        -eta_bins[0]
    )

    eta_tick_positions=(
        nice_etas
        -eta_bins[0]
    )/eta_step-0.5

    ax.set_yticks(
        eta_tick_positions
    )

    ax.set_yticklabels(
        [
            f"{eta_value:.1f}"
            for eta_value in nice_etas
        ]
    )

    eta_zero_position=(
        0.0
        -eta_bins[0]
    )/eta_step-0.5

    ax.axhline(
        eta_zero_position,
        color=CENTER_LINE_COLOR,
        linestyle=CENTER_LINE_STYLE,
        linewidth=CENTER_LINE_WIDTH,
        zorder=5
    )

    ax.set_xlabel(
        r"$\delta$",
        fontsize=LABEL_FONTSIZE,
        labelpad=9
    )

    ax.set_ylabel(
        r"$\eta=(Z_A-Z_B)/(Z_A+Z_B)$",
        fontsize=LABEL_FONTSIZE,
        labelpad=10
    )

    ax.set_xlim(
        x_edges[0],
        x_edges[-1]
    )

    ax.set_ylim(
        y_edges[0],
        y_edges[-1]
    )

    ax.set_aspect(
        "equal",
        adjustable="box"
    )

    divider=make_axes_locatable(
        ax
    )

    cax=divider.append_axes(
        "right",
        size="4%",
        pad=0.25
    )

    cbar=fig.colorbar(
        mesh,
        cax=cax,
        ticks=cbar_ticks
    )

    cbar.ax.set_yticklabels(
        [
            format_colorbar_tick(
                tick,
                vmax
            )
            for tick in cbar_ticks
        ]
    )

    cbar.minorticks_off()

    cbar.set_label(
        cbar_label,
        fontsize=LABEL_FONTSIZE,
        labelpad=26,
        rotation=270
    )

    style_axes_and_cbar(
        ax,
        cbar
    )

    # LEGENDA ORIGINAL, SEM W_T
    ax.text(
        0.98,
        0.975,
        metric_name,
        transform=ax.transAxes,
        fontsize=LABEL_FONTSIZE,
        color="white",
        va="top",
        ha="right",
        bbox=dict(
            facecolor=BOX_COLOR,
            edgecolor="none",
            alpha=1.0
        ),
        zorder=20
    )

    output_base=os.path.join(
        out_dir,
        output_name
    )

    fig.savefig(
        output_base+".png",
        dpi=600,
        bbox_inches="tight"
    )

    fig.savefig(
        output_base+".pdf",
        dpi=600,
        bbox_inches="tight"
    )

    fig.savefig(
        output_base+".eps",
        dpi=600,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    return (
        output_base,
        map_max,
        vmax,
        cbar_ticks
    )

def plot_lente3_media(
    db,
    case_name,
    mix_factor,
    data_file,
    out_dir,
    colorbar_max
):
    eta_bins=np.linspace(
        -1.0,
        1.0,
        21
    )

    (
        eta_bins,
        eta_centers,
        mean_map,
        samples_map
    )=project_peaks_to_delta_eta(
        db,
        eta_bins=eta_bins
    )

    # NOME ORIGINAL, SEM ACRESCENTAR W_T
    output_name=(
        f"Lente3_Projecao_Delta_vs_Eta_Media_{case_name}"
    )

    (
        output_base,
        map_max,
        scale_max,
        cbar_ticks
    )=plot_delta_eta_metric(
        db=db,
        eta_bins=eta_bins,
        eta_centers=eta_centers,
        metric_map=mean_map,

        # LEGENDA ORIGINAL
        metric_name=r"$\left\langle N_{\mathrm{res}}\right\rangle$",

        # LABEL ORIGINAL DA COLORBAR
        cbar_label="Resonances",

        output_name=output_name,
        out_dir=out_dir,
        forced_colorbar_max=colorbar_max
    )

    np.savez_compressed(
        output_base+"_dados.npz",
        data_file=data_file,
        contrast_in=case_name,
        mix_factor=mix_factor,

        # apenas metadado; não aparece no gráfico
        counting_source="ressonancias_malha",

        eta_bins=eta_bins,
        eta_centers=eta_centers,
        delta_vals=db["delta_vals"],

        # dados crus continuam sendo a média
        mean_map=mean_map,
        samples_map=samples_map,

        # só a escala de COR usa log(1+N)
        color_scale="log1p_norm_only",
        colorbar_min=0.0,
        colorbar_max=scale_max,
        colorbar_ticks=cbar_ticks,
        mean_resonance_data_max=map_max
    )

    return (
        mean_map,
        samples_map,
        output_base,
        map_max,
        scale_max,
        cbar_ticks
    )

def main():
    for case_name,case in CASES.items():
        checkpoint=case["checkpoint"]
        mix_factor=case["f"]
        colorbar_max=case["colorbar_max"]

        data_dir=os.path.dirname(
            checkpoint
        )

        out_dir=os.path.join(
            data_dir,
            f"esquema_analise_resultados_{case_name}"
        )

        os.makedirs(
            out_dir,
            exist_ok=True
        )

        print("\n"+"="*100)
        print(
            f"{case_name.upper()} | "
            f"f={mix_factor:.2f}"
        )
        print("="*100)

        print(
            f"Data file: {checkpoint}"
        )

        print(
            f"Output dir: {out_dir}"
        )

        print(
            "N_res source: ressonancias_malha"
        )

        print(
            "Map quantity: MEAN resonance count"
        )

        print(
            "Color normalization only: log(1 + N)"
        )

        try:
            db=load_checkpoint_vetorizado(
                checkpoint
            )

            total_curves=len(
                db["alpha"]
            )

            n_valid=int(
                np.sum(
                    np.isfinite(
                        db["n_peaks"]
                    )
                )
            )

            n_invalid=(
                total_curves
                -n_valid
            )

            valid_counts=db["n_peaks"][
                np.isfinite(
                    db["n_peaks"]
                )
            ]

            max_curve_count=(
                float(
                    np.max(
                        valid_counts
                    )
                )
                if len(valid_counts)
                else np.nan
            )

            print(
                f"Curves in checkpoint: {total_curves}"
            )

            print(
                f"Valid curves used in map: {n_valid}"
            )

            print(
                f"Timeout/error excluded: {n_invalid}"
            )

            print(
                f"Delta values: {len(db['delta_vals'])}"
            )

            print(
                "Maximum N_res in an individual W_T curve: "
                f"{max_curve_count:g}"
            )

            (
                mean_map,
                samples_map,
                output_base,
                map_max,
                scale_max,
                cbar_ticks
            )=plot_lente3_media(
                db=db,
                case_name=case_name,
                mix_factor=mix_factor,
                data_file=checkpoint,
                out_dir=out_dir,
                colorbar_max=colorbar_max
            )

            valid_cells=np.isfinite(
                mean_map
            )

            print(
                "\nProjection summary:"
            )

            print(
                "Completed valid pairs per bin: "
                f"{samples_map[valid_cells].min()} "
                f"to {samples_map[valid_cells].max()}"
            )

            print(
                "Minimum mean resonance count: "
                f"{np.nanmin(mean_map):.6g}"
            )

            print(
                "Maximum mean resonance count: "
                f"{map_max:.6g}"
            )

            print(
                "Colorbar range: "
                f"0 -> {scale_max:g}"
            )

            print(
                "Colorbar ticks: "
                f"{cbar_ticks}"
            )

            print(
                "\nSaved files:"
            )

            print(
                output_base+".png"
            )

            print(
                output_base+".pdf"
            )

            print(
                output_base+".eps"
            )

            print(
                output_base+"_dados.npz"
            )

        except FileNotFoundError:
            print(
                f"Error: data file not found:\n{checkpoint}"
            )

        except (
            KeyError,
            ValueError,
            RuntimeError
        ) as error:
            print(
                f"Error while generating the map:\n{error}"
            )

if __name__=="__main__":
    main()

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
BASE_DIR=DADOS
OUT_DIR=os.path.join(BASE_DIR,"esquema_analise_resultados_WT_malha")

CASES={
    "velocity":{
        "f":0.00,
        "checkpoint":os.path.join(BASE_DIR,"coreshell_lmax10_mixed_mix0.00","checkpoint_coreshell_lmax10_mixed_mix0.00.npz")
    },
    "mixed":{
        "f":0.50,
        "checkpoint":os.path.join(BASE_DIR,"coreshell_lmax10_mixed_mix0.50","checkpoint_coreshell_lmax10_mixed_mix0.50.npz")
    },
    "density":{
        "f":1.00,
        "checkpoint":os.path.join(BASE_DIR,"coreshell_lmax10_mixed_mix1.00","checkpoint_coreshell_lmax10_mixed_mix1.00.npz")
    }
}

MANUAL_REPRESENTATIVE_PAIRS={
    "eta_positive":{"alpha":0.006333333333333333,"beta":0.001},
    "eta_zero":{"alpha":0.008111111111111111,"beta":0.008111111111111111},
    "eta_negative":{"alpha":0.001,"beta":0.006333333333333333}
}

TABLE_DELTAS=[0.10,0.50,1.35]
TABLE_FIGSIZE=(6.4,4.5)
SPACER_WIDTH=0.013

ROW_LABELS=[
    "1st Peak\n"+r"($x_M$)",
    "1st Peak\nFWHM",
    "1st Peak\n"+r"$Q$-factor",
    "Macro-FSR\n"+r"($\Delta x_M$)",
    "Avg. FWHM\n"+r"($\mu\pm\sigma$) [$\times10^{-4}$]",
    "Avg. $Q$-factor\n"+r"($\mu\pm\sigma$)"
]

def eta_from_alpha_beta(alpha,beta):
    return (float(alpha)-float(beta))/(float(alpha)+float(beta))

def load_curves(checkpoint):
    if not os.path.exists(checkpoint):
        raise FileNotFoundError(f"Checkpoint não encontrado:\n{checkpoint}")
    with np.load(checkpoint,allow_pickle=True) as data:
        if "curves" not in data.files:
            raise KeyError(f"Chave 'curves' não encontrada:\n{checkpoint}")
        curves=data["curves"].item()

    rows=[]
    for key,item in curves.items():
        if item.get("status")!="ok":
            continue
        alpha=float(item.get("alpha",np.nan))
        beta=float(item.get("beta",np.nan))
        delta=float(item.get("delta",np.nan))
        if not (np.isfinite(alpha) and alpha>0 and np.isfinite(beta) and beta>0 and np.isfinite(delta)):
            continue
        rows.append({
            "curve_key":key,
            "alpha":alpha,
            "beta":beta,
            "eta":eta_from_alpha_beta(alpha,beta),
            "delta":delta,
            "item":item
        })
    return pd.DataFrame(rows)

def find_nearest_alpha_beta(df,target_alpha,target_beta):
    pairs=df[["alpha","beta"]].drop_duplicates().reset_index(drop=True)
    if len(pairs)==0:
        raise RuntimeError("Nenhum par alpha/beta válido.")
    target=np.log(np.array([target_alpha,target_beta],dtype=float))
    values=np.log(pairs.to_numpy(dtype=float))
    i=int(np.argmin(np.sum((values-target)**2,axis=1)))
    return float(pairs.loc[i,"alpha"]),float(pairs.loc[i,"beta"])

def select_nearest_delta_rows(df,alpha,beta,target_deltas):
    pair_df=df.loc[
        np.isclose(df["alpha"],alpha,rtol=1e-9,atol=0)
        & np.isclose(df["beta"],beta,rtol=1e-9,atol=0)
    ]
    rows=[]
    for target_delta in target_deltas:
        if len(pair_df)==0:
            rows.append(None)
            continue
        i=int(np.argmin(np.abs(pair_df["delta"].to_numpy(dtype=float)-target_delta)))
        rows.append(pair_df.iloc[i])
    return rows

def summarize_mesh_peaks(row):
    if row is None:
        return None

    peaks=row["item"].get("ressonancias_malha",[]) or []
    peaks=[p for p in peaks if np.isfinite(p.get("xM_peak",np.nan))]

    if not peaks:
        return None

    peaks=sorted(peaks,key=lambda p:float(p["xM_peak"]))
    x=np.asarray([p.get("xM_peak",np.nan) for p in peaks],dtype=float)
    fwhm=np.asarray([p.get("FWHM_energy",np.nan) for p in peaks],dtype=float)
    q=np.asarray([p.get("Q_energy",np.nan) for p in peaks],dtype=float)

    valid_fwhm=fwhm[np.isfinite(fwhm)]
    valid_q=q[np.isfinite(q)]
    gaps=np.diff(x) if len(x)>=2 else np.array([],dtype=float)

    return {
        "Actual_Delta":float(row["delta"]),
        "N_peaks":len(peaks),
        "First_Peak_xM":float(x[0]),
        "First_FWHM":float(fwhm[0]) if np.isfinite(fwhm[0]) else np.nan,
        "First_Q":float(q[0]) if np.isfinite(q[0]) else np.nan,
        "Macro_FSR_Median":float(np.median(gaps)) if len(gaps) else np.nan,
        "Mean_FWHM":float(np.mean(valid_fwhm)) if len(valid_fwhm) else np.nan,
        "Std_FWHM":float(np.std(valid_fwhm,ddof=1)) if len(valid_fwhm)>1 else (0.0 if len(valid_fwhm)==1 else np.nan),
        "Mean_Q":float(np.mean(valid_q)) if len(valid_q) else np.nan,
        "Std_Q":float(np.std(valid_q,ddof=1)) if len(valid_q)>1 else (0.0 if len(valid_q)==1 else np.nan)
    }

def latex_sci(value,sig=3):
    if not np.isfinite(value):
        return "--"
    value=float(value)
    if value==0:
        return "0"
    exponent=int(np.floor(np.log10(abs(value))))
    mantissa=value/10**exponent
    return rf"${mantissa:.{sig-1}f}\times10^{{{exponent}}}$"

def fmt_peak_x(value):
    return "--" if not np.isfinite(value) else f"{float(value):.5g}"

def fmt_fwhm(value):
    return latex_sci(value,3)

def fmt_q(value):
    if not np.isfinite(value):
        return "--"
    value=float(value)
    if abs(value)>=1e3 or (0<abs(value)<1e-2):
        return latex_sci(value,3)
    return f"{value:.3g}"

def fmt_macro_fsr(value):
    return latex_sci(value,3)

def fmt_mean_std(mean,std,decimals=2):
    if not np.isfinite(mean):
        return "--"
    if not np.isfinite(std):
        std=0.0
    return f"${mean:.{decimals}f}$\n"+rf"$\pm{std:.{decimals}f}$"

def fmt_mean_std_sci(mean,std,sig=2):
    if not np.isfinite(mean):
        return "--"
    if not np.isfinite(std):
        std=0.0

    mean_text=latex_sci(mean,sig)

    if std==0:
        std_text=r"$\pm0$"
    else:
        exponent=int(np.floor(np.log10(abs(std))))
        mantissa=std/10**exponent
        std_text=rf"$\pm{mantissa:.{sig-1}f}\times10^{{{exponent}}}$"

    return mean_text+"\n"+std_text

def make_table_data(summaries):
    return [
        [fmt_peak_x(s["First_Peak_xM"]) if s else "--" for s in summaries],
        [fmt_fwhm(s["First_FWHM"]) if s else "--" for s in summaries],
        [fmt_q(s["First_Q"]) if s else "--" for s in summaries],
        [fmt_macro_fsr(s["Macro_FSR_Median"]) if s else "--" for s in summaries],
        [fmt_mean_std(s["Mean_FWHM"]*1e4,s["Std_FWHM"]*1e4,2) if s else "--" for s in summaries],
        [fmt_mean_std_sci(s["Mean_Q"],s["Std_Q"],2) if s else "--" for s in summaries]
    ]

def save_table(filename,data):
    with plt.rc_context({
        "font.family":"STIXGeneral",
        "mathtext.fontset":"stix",
        "font.size":30,
        "figure.facecolor":"white",
        "axes.facecolor":"white",
        "savefig.facecolor":"white",
        "savefig.edgecolor":"white"
    }):
        fig,ax=plt.subplots(figsize=TABLE_FIGSIZE,facecolor="white")
        ax.axis("off")
        ax.set_facecolor("white")

        table_data=[
            [ROW_LABELS[i],data[i][0],"",data[i][1],"",data[i][2]]
            for i in range(len(ROW_LABELS))
        ]

        table=ax.table(
            cellText=table_data,
            colLabels=[
                "Parameter",
                r"$\delta=0.10$",
                "",
                r"$\delta=0.50$",
                "",
                r"$\delta=1.35$"
            ],
            cellLoc="center",
            colWidths=[0.36,0.20,SPACER_WIDTH,0.20,SPACER_WIDTH,0.20],
            bbox=[0.02,0.02,0.96,0.96]
        )

        table.auto_set_font_size(False)
        table.set_fontsize(20)

        spacer_cols=(2,4)
        value_cols=(1,3,5)

        for (row,col),cell in table.get_celld().items():
            cell.visible_edges=""
            cell.set_edgecolor("black")
            cell.set_linewidth(0.0)
            cell.set_facecolor("white")
            cell.PAD=0.01

            text=cell.get_text()
            text.set_color("black")
            text.set_fontweight("normal")
            text.set_va("center")
            text.set_linespacing(0.90)

            if col==0:
                text.set_ha("left")
                text.set_fontsize(19)
            elif col in spacer_cols:
                text.set_text("")
            else:
                text.set_ha("center")
                text.set_fontsize(20)

        for col in range(6):
            table[(0,col)].visible_edges="TB"
            table[(0,col)].set_linewidth(1.5)

        table[(0,0)].get_text().set_fontweight("bold")

        for col in range(6):
            table[(3,col)].visible_edges="B"
            table[(3,col)].set_linewidth(1.1)
            table[(6,col)].visible_edges="B"
            table[(6,col)].set_linewidth(1.5)

        for row in range(1,7):
            table[(row,0)].get_text().set_fontsize(19)

        for row in (1,3):
            for col in value_cols:
                table[(row,col)].get_text().set_fontsize(20)

        for col in value_cols:
            table[(2,col)].get_text().set_fontsize(18.2)
            table[(4,col)].get_text().set_fontsize(18.2)
            table[(5,col)].get_text().set_fontsize(19)
            table[(6,col)].get_text().set_fontsize(18.2)

        fig.subplots_adjust(left=0.005,right=0.995,bottom=0.005,top=0.995)

        fig.savefig(filename+".png",dpi=600,bbox_inches="tight",pad_inches=0.01,facecolor="white",edgecolor="white")
        fig.savefig(filename+".pdf",bbox_inches="tight",pad_inches=0.01,facecolor="white",edgecolor="white")
        fig.savefig(filename+".svg",bbox_inches="tight",pad_inches=0.01,facecolor="white",edgecolor="white")

        plt.show()
        plt.close(fig)

def main():
    for regime,case in CASES.items():
        print("\n"+"="*90)
        print(f"{regime.upper()} | f={case['f']:.2f}")
        print("="*90)

        curves_df=load_curves(case["checkpoint"])

        # MESMA PASTA USADA PELO CÓDIGO ANTERIOR
        regime_dir=os.path.join(OUT_DIR,regime)
        os.makedirs(regime_dir,exist_ok=True)

        for case_name,info in MANUAL_REPRESENTATIVE_PAIRS.items():
            alpha,beta=find_nearest_alpha_beta(
                curves_df,
                info["alpha"],
                info["beta"]
            )

            selected_rows=select_nearest_delta_rows(
                curves_df,
                alpha,
                beta,
                TABLE_DELTAS
            )

            summaries=[
                summarize_mesh_peaks(row)
                for row in selected_rows
            ]

            print(
                f"\n{case_name} | "
                f"alpha={alpha:.6g} | "
                f"beta={beta:.6g} | "
                f"eta={eta_from_alpha_beta(alpha,beta):+.4f}"
            )

            for target_delta,summary in zip(TABLE_DELTAS,summaries):
                if summary is None:
                    print(f"  delta={target_delta:.2f} | sem picos da malha")
                else:
                    print(
                        f"  delta={summary['Actual_Delta']:.2f} | "
                        f"N_picos_malha={summary['N_peaks']}"
                    )

            data=make_table_data(summaries)

            # MESMO NOME DO ARQUIVO ANTERIOR -> SUBSTITUI AUTOMATICAMENTE
            filename=os.path.join(
                regime_dir,
                f"table_{case_name}_compact"
            )

            save_table(filename,data)

            print(f"  SUBSTITUÍDO: {filename}.png")
            print(f"               {filename}.pdf")
            print(f"               {filename}.svg")

    print("\n"+"="*90)
    print("TABELAS SUBSTITUÍDAS")
    print("="*90)
    print(OUT_DIR)

if __name__=="__main__":
    main()